In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 1. Clone your collaboration repository
# Replace with your actual GitHub URL
#!git clone https://github.com/nmorok/Teleconnections-ViT.git
!git clone -b claude_code --single-branch https://github.com/nmorok/Teleconnections-ViT.git

# 2. Enter the repository directory
import os
os.chdir('Teleconnections-ViT')
# 3. Add the current directory to sys.path so 'import model' works
import sys
sys.path.append(os.getcwd())

# 4. Verify the files are present
print("Files in current directory:", os.listdir())

Cloning into 'Teleconnections-ViT'...
remote: Enumerating objects: 1215, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 1215 (delta 30), reused 35 (delta 11), pack-reused 1144 (from 1)
Receiving objects: 100% (1215/1215), 1.77 GiB | 40.39 MiB/s, done.
Resolving deltas: 100% (489/489), done.
Updating files: 100% (521/521), done.
Files in current directory: ['Pipeline.md', 'cell12_content.txt', 'grid_mask_verification.png', 'models', 'LICENSE', 'change_log.md', 'data', '.git', 'README.md', 'Methods.md', 'CrabTransformer_Methods.md', 'CLAUDE.md', 'notebook', 'requirements.txt', '.gitignore']


In [4]:
from google.colab import drive
import torch
import math


# 2. Define your data path (Update this to your actual Drive folder)
# Usually formatted as: /content/drive/MyDrive/Folder_Name
DATA_PATH = '/content/drive/MyDrive/Teleconnection_ViT/data'
REPO_DIR = '/content/Teleconnections-ViT'

# 3. Quick verification check
if os.path.exists(DATA_PATH):
    print(f"✓ Data folder found at: {DATA_PATH}")
    print("Files available:", os.listdir(DATA_PATH))
else:
    print(f"✗ ERROR: Could not find folder at {DATA_PATH}. Check your Drive path.")

# 4. Hardware Check
# 1. Check if CUDA (GPU support) is available
print(f"Is CUDA available? {torch.cuda.is_available()}")

# 2. Get the name of the GPU assigned by Colab Pro
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Using CPU. Check your Colab Runtime settings.")

✓ Data folder found at: /content/drive/MyDrive/Teleconnection_ViT/data
Files available: ['easy', 'medium', 'hard']
Is CUDA available? True
GPU Name: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [5]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import os
import math

from torch.utils.data import DataLoader
from models.model import CrabTransformer
from models.losses import TweedieLoss, MSELoss_cm
from data.data_helper import CrabDataset, get_dataloaders

The below is looking to make sure the transformations and the dataloaders aren't impacting the data.

I'll eventually look at the data post transformer model to make sure it looks similar

In [ ]:
"""
validate_configs.py
====================
Checks that every channel-config × prediction-mode combination in your run
matrix loads data correctly, builds the model, and completes a forward pass —
all on CPU, in seconds, with no training required.

What it does
------------
For each combination it:
  1. Calls get_dataloaders() and pulls ONE batch from the training loader.
  2. Verifies tensor shapes, channel count, and mask length.
  3. Instantiates CrabTransformer with the dataset's channel metadata.
  4. Runs one forward pass and checks output shape.
  5. Runs one forward pass with return_attention=True and checks attention shape.
  6. Prints a PASS / FAIL line with key numbers.

After testing all configs it:
  • Prints a summary table (all results at a glance).
  • Saves a figure for each tested data type (dummy / real) showing:
      - One row per channel config.
      - Columns = each channel in the input tensor, shown as a 50×50 heatmap
        (first sample, first channel group, mean across lookbacks for history
        channels to keep the grid manageable).

Output files
------------
  validation_results.csv   — pass/fail + shapes for every config
  validation_dummy.png     — channel grid for dummy data
  validation_real.png      — channel grid for real data  (if real data exists)

Usage
-----
  python validate_configs.py            # tests dummy only (safe if no real data)
  python validate_configs.py --real     # also test real data configs

All paths are relative to REPO_DIR; adjust the two constants at the top.
"""

import sys
import os
import argparse
import traceback
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd

# ---- adjust to your layout ----------------------------------------
REPO_DIR   = '/content/Teleconnections-ViT'
OUTPUT_DIR = '/content/drive/MyDrive/Teleconnection_ViT/validation'
# -------------------------------------------------------------------

sys.path.insert(0, REPO_DIR)
from data.data_helper  import get_dataloaders, get_channel_info
from models.model      import CrabTransformer


# ============================================================
#  CONFIG TABLES  (mirror train.py exactly)
# ============================================================

MODEL_CONFIGS = {
    'normal': {'embed_dim': 128, 'num_heads': 8, 'num_layers': 6,
               'd_ff': 512, 'dropout': 0.0},
    'small':  {'embed_dim': 128, 'num_heads': 4, 'num_layers': 3,
               'd_ff': 512, 'dropout': 0.0},
}

CHANNEL_CONFIGS = {
    'all':           {'use_spawners': True,  'use_recruits': True,  'use_temp': True},
    'sp_rec':        {'use_spawners': True,  'use_recruits': True,  'use_temp': False},
    'sp_temp':       {'use_spawners': True,  'use_recruits': False, 'use_temp': True},
    'rec_temp':      {'use_spawners': False, 'use_recruits': True,  'use_temp': True},
    'spawners_only': {'use_spawners': True,  'use_recruits': False, 'use_temp': False},
    'recruits_only': {'use_spawners': False, 'use_recruits': True,  'use_temp': False},
    'temp_only':     {'use_spawners': False, 'use_recruits': False, 'use_temp': True},
}

PREDICTION_MODES = {
    'normal':         {'incl_curr': True,  'lag': 0},
    'one_year_ahead': {'incl_curr': False, 'lag': 0},
    'lag5':           {'incl_curr': True,  'lag': 5},
}

# Human-readable channel names in tensor order.
# include_current mirrors data_helper.get_channel_info — when False,
# current-year spawner and temp channels are absent from the tensor.
def channel_names_for(use_spawners, use_recruits, use_temp,
                      include_current: bool = True):
    names = []
    if use_spawners:
        if include_current:
            names += ['Sp(t)']
        names += ['Sp(t-1)', 'Sp(t-2)', 'Sp(t-3)', 'Sp(t-4)', 'Sp(t-5)']
    if use_recruits:
        names += ['Rc(t-1)', 'Rc(t-2)', 'Rc(t-3)', 'Rc(t-4)', 'Rc(t-5)']
    if use_temp:
        if include_current:
            names += ['Tmp(t)']
        names += ['Tmp(t-1)', 'Tmp(t-2)', 'Tmp(t-3)', 'Tmp(t-4)', 'Tmp(t-5)']
    return names


# ============================================================
#  YEAR SPLITS
# ============================================================

def get_year_splits(data_type, lag):
    if data_type == 'real':
        return (24, 8, 4) if lag == 0 else (21, 6, 4)
    return (18, 9, 3)


# ============================================================
#  RESULT DATACLASS
# ============================================================

@dataclass
class ConfigResult:
    data_type:    str
    level:        str
    channel_cfg:  str
    pred_mode:    str
    model_size:   str
    passed:       bool
    error:        str          = ''
    in_channels:  int          = 0
    input_shape:  str          = ''
    output_shape: str          = ''
    mask_shape:   str          = ''
    attn_shape:   str          = ''
    mask_indices: str          = ''
    n_valid_yr:   int          = 0       # non-zero valid_year flags in batch
    input_mean:   float        = 0.0
    input_std:    float        = 0.0
    target_mean:  float        = 0.0
    sample_tensor: Optional[np.ndarray] = field(default=None, repr=False)
    channel_names: list        = field(default_factory=list, repr=False)


# ============================================================
#  SINGLE CONFIG TEST
# ============================================================

def test_config(data_type, level, channel_cfg, pred_mode,
                model_size='normal') -> ConfigResult:
    """
    Load one batch, run one forward pass, return a ConfigResult.
    Uses batch_size=2 and only reads the first batch — very fast on CPU.
    """
    result = ConfigResult(
        data_type=data_type, level=level, channel_cfg=channel_cfg,
        pred_mode=pred_mode, model_size=model_size, passed=False,
    )

    ccfg = CHANNEL_CONFIGS[channel_cfg]
    pcfg = PREDICTION_MODES[pred_mode]
    t_years, v_years, te_years = get_year_splits(data_type, pcfg['lag'])

    try:
        # -- Data --
        train_loader, _, _ = get_dataloaders(
            batch_size=2, memory_years=5,
            train_years=t_years, val_years=v_years, test_years=te_years,
            level=level, data_type=data_type,
            include_current_spawner=pcfg['incl_curr'],
            lag=pcfg['lag'],
            use_temp=ccfg['use_temp'],
            use_spawners=ccfg['use_spawners'],
            use_recruits=ccfg['use_recruits'],
        )

        ds = train_loader.dataset
        result.in_channels  = ds.in_channels
        result.mask_indices = str(ds.channel_mask_indices)
        result.channel_names = channel_names_for(
            ccfg['use_spawners'], ccfg['use_recruits'],
            ds.use_temp,           # actual temp (False for dummy)
            include_current=pcfg['incl_curr'],
        )

        # Pull ONE batch
        batch = next(iter(train_loader))
        inputs, targets, temporal_mask, year_idx, spatial_mask, valid_year = batch

        result.input_shape  = str(list(inputs.shape))
        result.mask_shape   = str(list(temporal_mask.shape))
        result.target_mean  = float(targets.mean())
        result.input_mean   = float(inputs.mean())
        result.input_std    = float(inputs.std())
        result.n_valid_yr   = int((valid_year > 0).sum())

        # Shape assertions
        B = inputs.shape[0]
        assert inputs.shape == (B, ds.in_channels, 50, 50), \
            f"Input shape mismatch: {inputs.shape}"
        assert targets.shape == (B, 1, 50, 50), \
            f"Target shape mismatch: {targets.shape}"
        assert temporal_mask.shape == (B, 6), \
            f"Temporal mask shape mismatch: {temporal_mask.shape}"
        assert len(ds.channel_mask_indices) == ds.in_channels, \
            f"channel_mask_indices length {len(ds.channel_mask_indices)} ≠ in_channels {ds.in_channels}"

        # -- Model --
        mcfg  = MODEL_CONFIGS[model_size]
        model = CrabTransformer(
            grid_size=50, patch_size=5,
            in_channels=ds.in_channels,
            embed_dim=mcfg['embed_dim'],
            num_heads=mcfg['num_heads'],
            num_layers=mcfg['num_layers'],
            d_ff=mcfg['d_ff'],
            dropout=mcfg['dropout'],
            channel_mask_indices=ds.channel_mask_indices,
        )
        model.eval()

        with torch.no_grad():
            # Normal forward
            output = model(inputs, year_idx, temporal_mask,
                           spatial_mask=spatial_mask)
            result.output_shape = str(list(output.shape))
            assert output.shape == (B, 1, 50, 50), \
                f"Output shape mismatch: {output.shape}"

            # With attention
            output_a, attn_maps = model(inputs, year_idx, temporal_mask,
                                        spatial_mask=spatial_mask,
                                        return_attention=True)
            assert len(attn_maps) == mcfg['num_layers'], \
                f"Expected {mcfg['num_layers']} attention maps, got {len(attn_maps)}"
            expected_heads = mcfg['num_heads']
            assert attn_maps[0].shape[1] == expected_heads, \
                f"Attention head count mismatch"
            result.attn_shape = str(list(attn_maps[0].shape))

            # Output is non-negative (softplus activated)
            assert float(output.min()) >= 0.0, "Output contains negative values"

            # Masked cells (outside EBS region) should be exactly zero.
            # Guard against dummy data where spatial_mask is all-ones
            # — indexing an empty selection with .max() raises an error.
            outside = (spatial_mask == 0).unsqueeze(1)
            if outside.any():
                assert float(output[outside].max()) == 0.0, \
                    "Non-zero values found outside spatial mask"

        # Store the raw input tensor for visualisation (first sample)
        result.sample_tensor = inputs[0].numpy()   # [C, 50, 50]
        result.passed        = True

    except Exception as e:
        result.error = f"{type(e).__name__}: {e}"
        # Print traceback for debugging
        traceback.print_exc()

    return result


# ============================================================
#  VISUALISATION  — channel grid for a set of results
# ============================================================

CMAP_BY_GROUP = {
    'Sp': 'YlOrRd',
    'Rc': 'Blues',
    'Tm': 'RdBu_r',
}

def group_color(name):
    if name.startswith('Sp'):   return CMAP_BY_GROUP['Sp']
    if name.startswith('Rc'):   return CMAP_BY_GROUP['Rc']
    return CMAP_BY_GROUP['Tm']


def make_channel_figure(results: list, title: str, save_path: str):
    """
    Grid figure: one row per channel config, one column per channel.
    The widest config (most channels) determines the number of columns.
    Narrower configs leave trailing cells blank.
    Shows the first sample's spatial grid for each channel.
    """
    # Only passed results
    passed = [r for r in results if r.passed and r.sample_tensor is not None]
    if not passed:
        print("  No passed results with tensors — skipping figure.")
        return

    max_ch   = max(r.in_channels for r in passed)
    n_rows   = len(passed)
    n_cols   = max_ch + 1    # +1 for the row-label column
    fig_w    = min(2.2 * n_cols + 1.5, 40)
    fig_h    = min(2.4 * n_rows + 1.0, 60)

    fig = plt.figure(figsize=(fig_w, fig_h), dpi=100)
    fig.suptitle(title, fontsize=12, fontweight='bold', y=1.01)

    gs = gridspec.GridSpec(
        n_rows, n_cols,
        figure=fig,
        hspace=0.55, wspace=0.08,
        left=0.01, right=0.99, top=0.97, bottom=0.03,
    )

    for row_idx, r in enumerate(passed):
        ch_names = r.channel_names
        tensor   = r.sample_tensor          # [C, 50, 50]

        # Row label (left-most column, spanning the full height)
        ax_lbl = fig.add_subplot(gs[row_idx, 0])
        mode_short = {'normal': 'norm', 'one_year_ahead': '1yr↑', 'lag5': 'lag5'}
        lbl = (f"{r.channel_cfg}\n"
               f"{mode_short.get(r.pred_mode, r.pred_mode)}\n"
               f"{r.model_size}\n"
               f"[{r.in_channels}ch]")
        ax_lbl.text(0.5, 0.5, lbl, ha='center', va='center',
                    fontsize=7, fontweight='bold', transform=ax_lbl.transAxes,
                    multialignment='center')
        ax_lbl.axis('off')

        # One subplot per channel
        for c_idx in range(max_ch):
            ax = fig.add_subplot(gs[row_idx, c_idx + 1])

            if c_idx < r.in_channels:
                img  = tensor[c_idx]
                name = ch_names[c_idx] if c_idx < len(ch_names) else f'ch{c_idx}'
                cmap = group_color(name)
                vmin, vmax = float(img.min()), float(img.max())
                if vmax - vmin < 1e-6:
                    vmin, vmax = 0.0, 1.0
                ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax,
                          interpolation='nearest', aspect='equal')
                ax.set_title(name, fontsize=6, pad=2)

                # Show temporal mask slot
                midx = r.sample_tensor.shape[0]   # just a fallback
                if hasattr(r, 'mask_indices') and r.mask_indices:
                    try:
                        mlist = eval(r.mask_indices)
                        if c_idx < len(mlist):
                            midx = mlist[c_idx]
                            ax.set_xlabel(f'm[{midx}]', fontsize=5, labelpad=1)
                    except Exception:
                        pass

                # Frame colour by channel group
                edge = {'Sp': '#d62728', 'Rc': '#1f77b4', 'Tm': '#2ca02c'}
                grp  = name[:2] if len(name) >= 2 else '??'
                for sp in ax.spines.values():
                    sp.set_edgecolor(edge.get(grp, 'grey'))
                    sp.set_linewidth(1.5)
            else:
                # Empty cell for configs with fewer channels
                ax.set_facecolor('#f0f0f0')
                for sp in ax.spines.values():
                    sp.set_visible(False)

            ax.set_xticks([])
            ax.set_yticks([])

    # Legend
    from matplotlib.patches import Patch
    legend_els = [
        Patch(facecolor='#d62728', label='Spawner channels'),
        Patch(facecolor='#1f77b4', label='Recruit channels'),
        Patch(facecolor='#2ca02c', label='Temperature channels'),
        Patch(facecolor='#f0f0f0', label='Unused slot'),
    ]
    fig.legend(handles=legend_els, loc='lower center',
               ncol=4, fontsize=8, frameon=True,
               bbox_to_anchor=(0.5, -0.01))

    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.close()
    print(f"  Figure saved → {save_path}")


# ============================================================
#  SUMMARY TABLE
# ============================================================

def print_summary_table(results: list):
    rows = []
    for r in results:
        status = '✅ PASS' if r.passed else f'❌ FAIL'
        rows.append({
            'status':      status,
            'data':        r.data_type[:4],
            'level':       r.level[:5],
            'channel_cfg': r.channel_cfg,
            'pred_mode':   r.pred_mode,
            'model':       r.model_size,
            'in_ch':       r.in_channels,
            'input_shape': r.input_shape,
            'output':      r.output_shape,
            'attn':        r.attn_shape,
            'n_valid':     r.n_valid_yr,
            'inp_mean':    f'{r.input_mean:.3f}',
            'tgt_mean':    f'{r.target_mean:.3f}',
            'error':       r.error[:60] if r.error else '',
        })
    df = pd.DataFrame(rows)

    # Wide print
    pd.set_option('display.max_rows', 300)
    pd.set_option('display.max_columns', 20)
    pd.set_option('display.width', 220)
    pd.set_option('display.max_colwidth', 65)

    n_pass = sum(r.passed for r in results)
    n_fail = len(results) - n_pass

    print(f"\n{'='*120}")
    print(f"  VALIDATION SUMMARY  —  {n_pass} passed, {n_fail} failed "
          f"out of {len(results)} tested")
    print(f"{'='*120}")
    print(df.to_string(index=False))

    if n_fail > 0:
        print(f"\n{'─'*120}")
        print("FAILURES:")
        for r in results:
            if not r.passed:
                print(f"  {r.data_type}/{r.level}/{r.channel_cfg}/"
                      f"{r.pred_mode}/{r.model_size}  →  {r.error}")
    print(f"{'='*120}\n")

    pd.reset_option('display.max_rows')
    pd.reset_option('display.max_columns')
    pd.reset_option('display.width')
    return df


# ============================================================
#  MAIN
# ============================================================

def main(test_real: bool = False):
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    all_results = []

    # ---- Build test matrix ----
    # One model size (normal) is enough to validate data pipeline.
    # Both model sizes are tested for the architecture checks.
    # We use a single dummy level (easy) for speed.

    test_configs = []

    # Dummy: all no-temp channel configs × all pred modes.
    # one_year_ahead is skipped only for recruits_only (no current-year channel).
    for ch_cfg, ccfg in CHANNEL_CONFIGS.items():
        if ccfg['use_temp']:
            continue     # no temp files for dummy
        for pred_mode, pcfg in PREDICTION_MODES.items():
            if pred_mode == 'lag5':
                continue   # no lag5 splits for dummy
            if pred_mode == 'one_year_ahead' and \
                    not (ccfg['use_spawners'] or ccfg['use_temp']):
                continue   # no-op: recruits_only has no current-year channel without spawners
            # Test with both model sizes
            for model_size in MODEL_CONFIGS:
                test_configs.append(dict(
                    data_type='dummy', level='easy',
                    channel_cfg=ch_cfg, pred_mode=pred_mode,
                    model_size=model_size,
                ))

    # Real: all 7 channel configs × all applicable modes × both model sizes
    if test_real:
        for ch_cfg, ccfg in CHANNEL_CONFIGS.items():
            for pred_mode, pcfg in PREDICTION_MODES.items():
                if pred_mode == 'one_year_ahead' and \
                        not (ccfg['use_spawners'] or ccfg['use_temp']):
                    continue   # no-op: recruits_only has no current-year channel
                for model_size in MODEL_CONFIGS:
                    test_configs.append(dict(
                        data_type='real', level='real',
                        channel_cfg=ch_cfg, pred_mode=pred_mode,
                        model_size=model_size,
                    ))

    n = len(test_configs)
    print(f"\n{'='*72}")
    print(f"  CrabTransformer  —  Config Validator")
    print(f"  {n} configurations to test  |  CPU-only, no training")
    print(f"  Output: {OUTPUT_DIR}")
    print(f"{'='*72}\n")

    for i, cfg in enumerate(test_configs, 1):
        label = (f"{cfg['data_type']}/{cfg['level']}/"
                 f"{cfg['channel_cfg']}/{cfg['pred_mode']}/{cfg['model_size']}")
        print(f"[{i:3d}/{n}]  Testing: {label} … ", end='', flush=True)
        result = test_config(**cfg)
        status = '✅' if result.passed else f'❌  {result.error[:80]}'
        print(status)
        all_results.append(result)

    # ---- Summary table ----
    df_summary = print_summary_table(all_results)

    # Save CSV
    csv_path = os.path.join(OUTPUT_DIR, 'validation_results.csv')
    # Drop the large tensor field before saving
    df_summary.to_csv(csv_path, index=False)
    print(f"Results saved → {csv_path}\n")

    # ---- Channel grid figures ----
    print("Generating channel grid figures …")

    # Dummy figure: one row per channel_cfg × pred_mode combo,
    # using 'normal' model size (both sizes test identically for data)
    dummy_results = [r for r in all_results
                     if r.data_type == 'dummy' and r.model_size == 'normal']
    if dummy_results:
        make_channel_figure(
            dummy_results,
            title='Dummy data — channel layout per config  '
                  '(red=spawner  blue=recruit  green=temp)',
            save_path=os.path.join(OUTPUT_DIR, 'validation_dummy.png'),
        )

    real_results = [r for r in all_results
                    if r.data_type == 'real' and r.model_size == 'normal']
    if real_results:
        make_channel_figure(
            real_results,
            title='Real data — channel layout per config  '
                  '(red=spawner  blue=recruit  green=temp)',
            save_path=os.path.join(OUTPUT_DIR, 'validation_real.png'),
        )

    # ---- Quick stats per channel config (data perspective) ----
    print("\n=== PER-CONFIG CHANNEL STATS (first batch, first sample) ===")
    seen = set()
    for r in all_results:
        key = (r.data_type, r.level, r.channel_cfg, r.pred_mode)
        if not r.passed or key in seen or r.sample_tensor is None:
            continue
        seen.add(key)
        tensor = r.sample_tensor   # [C, 50, 50]
        print(f"\n  {r.data_type}/{r.channel_cfg}/{r.pred_mode}  "
              f"[{r.in_channels} channels]")
        print(f"  {'Channel':<12}  {'mask_idx':>8}  "
              f"{'min':>7}  {'mean':>7}  {'max':>7}  "
              f"{'zeros%':>7}  {'nonzero_mean':>12}")
        mask_idx = eval(r.mask_indices)
        for c_idx in range(tensor.shape[0]):
            img = tensor[c_idx]
            mn, mu, mx = img.min(), img.mean(), img.max()
            z_pct = (img == 0).mean() * 100
            nz_mu = img[img != 0].mean() if (img != 0).any() else 0.0
            name  = r.channel_names[c_idx] if c_idx < len(r.channel_names) else f'ch{c_idx}'
            midx  = mask_idx[c_idx] if c_idx < len(mask_idx) else '?'
            print(f"  {name:<12}  {midx:>8}  "
                  f"{mn:>7.3f}  {mu:>7.3f}  {mx:>7.3f}  "
                  f"{z_pct:>6.1f}%  {nz_mu:>12.3f}")

    n_pass = sum(r.passed for r in all_results)
    n_fail = len(all_results) - n_pass
    print(f"\n{'='*72}")
    if n_fail == 0:
        print(f"  ✅  All {n_pass} configurations passed.")
    else:
        print(f"  ⚠   {n_pass} passed, {n_fail} FAILED.  "
              f"Check the table above for error details.")
    print(f"{'='*72}\n")


# ============================================================
#  ENTRY POINT
# ============================================================

if __name__ == '__main__':
    main(test_real=True)

In [ ]:
!pip install sympy==1.13.3 --quiet

In [9]:
"""
train.py  —  Master training script for all CrabTransformer configurations.

Directory layout
----------------
model_outputs/
  {model_size}/               normal | small
    {level}/                  easy | medium | hard | real
      {channel_cfg}/          all | sp_rec | sp_temp | rec_temp |
                              spawners_only | recruits_only | temp_only
        {pred_mode}/          normal | one_year_ahead | lag5
          {criterion}/        MSE | Tweedie
            best_model.pt
            training_history.json
            training_curves.png

Channel configurations (7 total)
----------------------------------
  all           spawners (6ch) + recruits (5ch) + temp (6ch)  = 17ch
  sp_rec        spawners + recruits                            = 11ch
  sp_temp       spawners + temp                               = 12ch
  rec_temp      recruits + temp                               = 11ch
  spawners_only spawners only                                 =  6ch
  recruits_only recruits only                                 =  5ch
  temp_only     temp only                                     =  6ch

Run matrix rules
----------------
* Dummy (easy/medium/hard): no temp files → skip configs that need temp.
* one_year_ahead: only valid when spawners are in the config (otherwise the
  mode has no effect — channel 0 doesn't exist to zero out).
* lag5: only valid for real data (separate split directories exist for lag=5).
* SKIP_IF_EXISTS=True  lets you resume an interrupted run safely.
"""

import os
import sys
import json
import traceback

import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ---- adjust to your Colab / Drive layout ----
REPO_DIR   = '/content/Teleconnections-ViT'
DRIVE_BASE = '/content/drive/MyDrive/Teleconnection_ViT/model_outputs'
# ---------------------------------------------

sys.path.insert(0, REPO_DIR)
from models.model     import CrabTransformer
from data.data_helper import get_dataloaders
from models.losses    import TweedieLoss, MSELoss_cm


# ============================================================
#  GLOBAL TRAINING HYPERPARAMETERS
# ============================================================

SKIP_IF_EXISTS  = True
NUM_EPOCHS      = 20
BATCH_SIZE      = 8
MEMORY_YEARS    = 5
TWEEDIE_POWER   = 1.2
MAX_LR          = 3e-4
BASE_LR         = 1e-4
WEIGHT_DECAY    = 1e-4
GRAD_CLIP       = 1.0
PATIENCE        = 20


# ============================================================
#  RUN MATRIX DEFINITIONS
# ============================================================

MODEL_CONFIGS = {
    'normal': {'embed_dim': 128, 'num_heads': 8, 'num_layers': 6,
               'd_ff': 512, 'dropout': 0.1},
    'small':  {'embed_dim': 128, 'num_heads': 4, 'num_layers': 3,
               'd_ff': 512, 'dropout': 0.2},
}

# Boolean flags only — in_channels and channel_mask_indices are derived
# automatically from the dataset after get_dataloaders() is called.
CHANNEL_CONFIGS = {
    'all':           {'use_spawners': True,  'use_recruits': True,  'use_temp': True},
    'sp_rec':        {'use_spawners': True,  'use_recruits': True,  'use_temp': False},
    'sp_temp':       {'use_spawners': True,  'use_recruits': False, 'use_temp': True},
    'rec_temp':      {'use_spawners': False, 'use_recruits': True,  'use_temp': True},
    'spawners_only': {'use_spawners': True,  'use_recruits': False, 'use_temp': False},
    'recruits_only': {'use_spawners': False, 'use_recruits': True,  'use_temp': False},
    'temp_only':     {'use_spawners': False, 'use_recruits': False, 'use_temp': True},
}

PREDICTION_MODES = {
    'normal':         {'incl_curr': True,  'lag': 0},
    'one_year_ahead': {'incl_curr': False, 'lag': 0},
    'lag5':           {'incl_curr': True,  'lag': 5},
}

CRITERIA = ['MSE', 'Tweedie']


# ============================================================
#  RUN MATRIX BUILDER
# ============================================================

def _needs_temp(ch):    return CHANNEL_CONFIGS[ch]['use_temp']
def _has_spawners(ch):  return CHANNEL_CONFIGS[ch]['use_spawners']
def _has_current_year_channel(ch):
    # one_year_ahead drops current-year channels (spawner + temp).
    # Only recruits_only has no current-year channel in either mode,
    # making one_year_ahead a no-op for it.
    ccfg = CHANNEL_CONFIGS[ch]
    return ccfg['use_spawners'] or ccfg['use_temp']


def build_run_matrix():
    runs = []

    # ---- Dummy: easy / medium / hard ----
    for level in ['easy', 'medium', 'hard']:
        for model_size in MODEL_CONFIGS:
            for ch_cfg in CHANNEL_CONFIGS:
                if _needs_temp(ch_cfg):
                    continue
                # Only run normal and one_year_ahead for dummy data
                for pred_mode in ['normal', 'one_year_ahead']:
                    if pred_mode == 'one_year_ahead' and not _has_current_year_channel(ch_cfg):
                        continue
                    for criterion in CRITERIA:
                        runs.append(dict(model_size=model_size, level=level,
                                         data_type='dummy', channel_cfg=ch_cfg,
                                         pred_mode=pred_mode, criterion=criterion))

    # ---- Real data: all 7 channel configs × all applicable modes ----
    for model_size in MODEL_CONFIGS:
        for ch_cfg in CHANNEL_CONFIGS:
            for pred_mode in PREDICTION_MODES:
                if pred_mode == 'one_year_ahead' and not _has_current_year_channel(ch_cfg):
                    continue
                for criterion in CRITERIA:
                    runs.append(dict(model_size=model_size, level='real',
                                     data_type='real', channel_cfg=ch_cfg,
                                     pred_mode=pred_mode, criterion=criterion))
    return runs


# ============================================================
#  HELPERS
# ============================================================

def get_year_splits(data_type, lag):
    if data_type == 'real':
        return (24, 8, 4) if lag == 0 else (21, 6, 4)
    return (18, 9, 3)   # dummy data — no lag5 splits exist

def get_run_dir(model_size, level, channel_cfg, pred_mode, criterion):
    return os.path.join(DRIVE_BASE, model_size, level, channel_cfg, pred_mode, criterion)


def compute_bias_correction(model, train_loader, device):
    """Full lognormal bias correction exp(μ + σ²/2) from training residuals."""
    model.eval()
    residuals = []
    with torch.no_grad():
        for inputs, targets, temporal_mask, year_idx, spatial_mask, valid_year in train_loader:
            inputs        = inputs.to(device)
            targets       = targets.to(device)
            temporal_mask = temporal_mask.to(device)
            year_idx      = year_idx.to(device)
            spatial_mask  = spatial_mask.to(device)
            outputs = model(inputs, year_idx, temporal_mask, spatial_mask=spatial_mask)
            for i in range(targets.shape[0]):
                if valid_year[i] > 0:
                    vm  = spatial_mask[i] > 0
                    res = (targets[i, 0][vm] - outputs[i, 0][vm]).cpu().numpy()
                    residuals.append(res)
    residuals = np.concatenate(residuals)
    mu        = float(residuals.mean())
    sigma2    = float(residuals.var())
    return float(np.exp(mu + 0.5 * sigma2)), mu, sigma2


def save_training_curves(history, run_dir, title_str):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.plot(epochs, history['train_loss'], color='steelblue', lw=2, label='Train')
    ax1.plot(epochs, history['val_loss'],   color='tomato',    lw=2, label='Val')
    ax1.set_title(title_str, fontsize=9); ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.plot(epochs, history['lr'], color='seagreen', lw=2)
    ax2.set_title('OneCycleLR'); ax2.set_xlabel('Epoch')
    ax2.set_ylabel('LR'); ax2.set_yscale('log'); ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(run_dir, 'training_curves.png'), dpi=150, bbox_inches='tight')
    plt.close()


# ============================================================
#  SINGLE RUN
# ============================================================

def train_one_run(model_size, level, data_type, channel_cfg, pred_mode, criterion):
    run_dir         = get_run_dir(model_size, level, channel_cfg, pred_mode, criterion)
    checkpoint_path = os.path.join(run_dir, 'best_model.pt')

    if SKIP_IF_EXISTS and os.path.exists(checkpoint_path):
        print(f"  ⏭  Skipping: {os.path.relpath(run_dir, DRIVE_BASE)}")
        return True

    os.makedirs(run_dir, exist_ok=True)
    mcfg = MODEL_CONFIGS[model_size]
    ccfg = CHANNEL_CONFIGS[channel_cfg]
    pcfg = PREDICTION_MODES[pred_mode]
    t_years, v_years, te_years = get_year_splits(data_type, pcfg['lag'])

    title_str = f"{model_size} | {level} | {channel_cfg} | {pred_mode} | {criterion}"
    print(f"\n{'='*72}\n  {title_str}\n  → {run_dir}\n{'='*72}")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"  Device: {device}")

    # -- Data --
    train_loader, val_loader, test_loader = get_dataloaders(
        batch_size=BATCH_SIZE, memory_years=MEMORY_YEARS,
        train_years=t_years, val_years=v_years, test_years=te_years,
        level=level, data_type=data_type,
        include_current_spawner=pcfg['incl_curr'],
        lag=pcfg['lag'],
        use_temp=ccfg['use_temp'],
        use_spawners=ccfg['use_spawners'],
        use_recruits=ccfg['use_recruits'],
    )

    # Read channel metadata directly from the dataset object
    ds                   = train_loader.dataset
    in_channels          = ds.in_channels
    channel_mask_indices = ds.channel_mask_indices
    print(f"  in_channels={in_channels}  mask_indices={channel_mask_indices}")

    # -- Model --
    model = CrabTransformer(
        grid_size=50, patch_size=5,
        in_channels=in_channels,
        embed_dim=mcfg['embed_dim'],
        num_heads=mcfg['num_heads'],
        num_layers=mcfg['num_layers'],
        d_ff=mcfg['d_ff'],
        dropout=mcfg['dropout'],
        channel_mask_indices=channel_mask_indices,
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable parameters: {n_params:,}")

    # Decoder bias warm-start
    all_log_targets = []
    for _, targets, _, _, spatial_mask, valid_year in train_loader:
        for i in range(targets.shape[0]):
            if valid_year[i] > 0:
                all_log_targets.append(targets[i, 0][spatial_mask[i] > 0].numpy())
    mean_log = float(np.concatenate(all_log_targets).mean())
    nn.init.constant_(model.decoder.conv_out.bias, mean_log)
    print(f"  Decoder bias warm-started at {mean_log:.4f}")

    # -- Loss / optimiser / scheduler --
    criterion_fn = TweedieLoss(power=TWEEDIE_POWER) if criterion == 'Tweedie' else MSELoss_cm()
    optimizer    = torch.optim.AdamW(model.parameters(), lr=BASE_LR,
                                     weight_decay=WEIGHT_DECAY)
    scheduler    = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=MAX_LR, epochs=NUM_EPOCHS,
        steps_per_epoch=len(train_loader), pct_start=0.3,
        anneal_strategy='cos', div_factor=25, final_div_factor=1000,
    )

    # -- Training loop --
    best_val_loss    = float('inf')
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': [], 'lr': []}

    for epoch in range(NUM_EPOCHS):
        model.train()
        epoch_loss = 0.0
        for inputs, targets, temporal_mask, year_idx, spatial_mask, valid_year in train_loader:
            inputs, targets   = inputs.to(device),       targets.to(device)
            temporal_mask     = temporal_mask.to(device)
            year_idx          = year_idx.to(device)
            spatial_mask      = spatial_mask.to(device)
            valid_year        = valid_year.to(device)

            optimizer.zero_grad()
            outputs = model(inputs, year_idx, temporal_mask, spatial_mask=spatial_mask)

            if criterion == 'Tweedie':
                loss = criterion_fn(torch.expm1(outputs).clamp(min=1e-6),
                                    torch.expm1(targets).clamp(min=1e-6),
                                    mask=spatial_mask, sample_mask=valid_year)
            else:
                loss = criterion_fn(outputs, targets,
                                    mask=spatial_mask, sample_mask=valid_year)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

        avg_train = epoch_loss / len(train_loader)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, targets, temporal_mask, year_idx, spatial_mask, valid_year in val_loader:
                inputs, targets   = inputs.to(device),       targets.to(device)
                temporal_mask     = temporal_mask.to(device)
                year_idx          = year_idx.to(device)
                spatial_mask      = spatial_mask.to(device)
                valid_year        = valid_year.to(device)
                outputs = model(inputs, year_idx, temporal_mask, spatial_mask=spatial_mask)
                if criterion == 'Tweedie':
                    val_loss += criterion_fn(torch.expm1(outputs).clamp(min=1e-6),
                                             torch.expm1(targets).clamp(min=1e-6),
                                             mask=spatial_mask,
                                             sample_mask=valid_year).item()
                else:
                    val_loss += criterion_fn(outputs, targets,
                                             mask=spatial_mask,
                                             sample_mask=valid_year).item()

        avg_val    = val_loss / len(val_loader)
        current_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)
        history['lr'].append(current_lr)

        improved = ''
        if avg_val < best_val_loss:
            best_val_loss    = avg_val
            patience_counter = 0
            torch.save(model.state_dict(), checkpoint_path)
            improved = '  ✓ saved'
        else:
            patience_counter += 1

        print(f"  Ep {epoch+1:3d}/{NUM_EPOCHS}  train={avg_train:.4f}  "
              f"val={avg_val:.4f}  lr={current_lr:.2e}  "
              f"pat={patience_counter}/{PATIENCE}{improved}")

        if patience_counter >= PATIENCE:
            print(f"  Early stop at epoch {epoch+1}")
            break

    # -- Bias correction (MSE only) --
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    history.update({'bias_correction': 1.0, 'bias_mu': 0.0, 'bias_sigma2': 0.0})
    if criterion == 'MSE':
        bc, mu, sigma2 = compute_bias_correction(model, train_loader, device)
        history.update({'bias_correction': bc, 'bias_mu': mu, 'bias_sigma2': sigma2})
        print(f"  Bias correction: {bc:.4f}  (μ={mu:.4f}, σ²={sigma2:.4f})")

    # -- Test loss --
    test_loss = 0.0
    with torch.no_grad():
        for inputs, targets, temporal_mask, year_idx, spatial_mask, valid_year in test_loader:
            inputs, targets   = inputs.to(device),       targets.to(device)
            temporal_mask     = temporal_mask.to(device)
            year_idx          = year_idx.to(device)
            spatial_mask      = spatial_mask.to(device)
            valid_year        = valid_year.to(device)
            outputs = model(inputs, year_idx, temporal_mask, spatial_mask=spatial_mask)
            if criterion == 'Tweedie':
                test_loss += criterion_fn(torch.expm1(outputs).clamp(min=1e-6),
                                          torch.expm1(targets).clamp(min=1e-6),
                                          mask=spatial_mask,
                                          sample_mask=valid_year).item()
            else:
                test_loss += criterion_fn(outputs, targets,
                                          mask=spatial_mask,
                                          sample_mask=valid_year).item()
    history['test_loss'] = test_loss / len(test_loader)
    print(f"  Test loss: {history['test_loss']:.4f}")

    # Save channel metadata in JSON for eval script to recover without re-loading data
    history['channel_cfg_meta'] = {
        'in_channels':          in_channels,
        'channel_mask_indices': channel_mask_indices,
        'use_spawners':         ccfg['use_spawners'],
        'use_recruits':         ccfg['use_recruits'],
        'use_temp':             ccfg['use_temp'],
        'incl_curr':            pcfg['incl_curr'],
        'lag':                  int(pcfg['lag']),
        'embed_dim':            mcfg['embed_dim'],
        'num_heads':            mcfg['num_heads'],
        'num_layers':           mcfg['num_layers'],
        'd_ff':                 mcfg['d_ff'],
    }

    with open(os.path.join(run_dir, 'training_history.json'), 'w') as f:
        json.dump(history, f, indent=2)

    save_training_curves(history, run_dir, title_str)
    print(f"  ✅  Done.  Best val loss: {best_val_loss:.4f}")
    return True


# ============================================================
#  ENTRY POINT
# ============================================================

if __name__ == '__main__':
    runs = build_run_matrix()
    n    = len(runs)

    print(f"\n{'='*72}")
    print(f"  CrabTransformer — Master Training Run")
    print(f"  Total planned runs: {n}  |  SKIP_IF_EXISTS={SKIP_IF_EXISTS}")
    print(f"  Output root: {DRIVE_BASE}")
    print(f"{'='*72}")
    for i, r in enumerate(runs, 1):
        print(f"  [{i:3d}/{n}]  {r['model_size']:6s}  {r['level']:6s}  "
              f"{r['channel_cfg']:14s}  {r['pred_mode']:15s}  {r['criterion']}")

    print(f"\nStarting …\n")
    failed = []
    for i, run in enumerate(runs, 1):
        print(f"\n[RUN {i}/{n}]")
        try:
            train_one_run(**run)
        except Exception as e:
            print(f"  ❌  {e}")
            traceback.print_exc()
            failed.append({'run': i, 'config': run, 'error': str(e)})

    print(f"\n{'='*72}")
    print(f"Training complete.  {n - len(failed)}/{n} runs succeeded.")
    if failed:
        print("Failed runs:")
        for f in failed:
            print(f"  [{f['run']}]  {f['config']}  →  {f['error']}")
    print(f"{'='*72}\n")


  CrabTransformer — Master Training Run
  Total planned runs: 140  |  SKIP_IF_EXISTS=True
  Output root: /content/drive/MyDrive/Teleconnection_ViT/model_outputs
  [  1/140]  normal  easy    sp_rec          normal           MSE
  [  2/140]  normal  easy    sp_rec          normal           Tweedie
  [  3/140]  normal  easy    sp_rec          one_year_ahead   MSE
  [  4/140]  normal  easy    sp_rec          one_year_ahead   Tweedie
  [  5/140]  normal  easy    spawners_only   normal           MSE
  [  6/140]  normal  easy    spawners_only   normal           Tweedie
  [  7/140]  normal  easy    spawners_only   one_year_ahead   MSE
  [  8/140]  normal  easy    spawners_only   one_year_ahead   Tweedie
  [  9/140]  normal  easy    recruits_only   normal           MSE
  [ 10/140]  normal  easy    recruits_only   normal           Tweedie
  [ 11/140]  small   easy    sp_rec          normal           MSE
  [ 12/140]  small   easy    sp_rec          normal           Tweedie
  [ 13/140]  small   e

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import os
from models.model import CrabTransformer
from data.data_helper import get_dataloaders

# Ensure you import your custom losses!
# from utils.losses import TweedieLoss, MSELoss_cm

def test_overfit_on_real_data(data_type='real', level='real', loss_criterion='MSE', lag=0):
    print("=" * 70)
    print(f"OVERFITTING TEST ({data_type}/{level}, {loss_criterion}, LAG={lag})")
    print("=" * 70)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # ---> FIX: Dynamically set splits based on lag <---
    if data_type == 'real':
        if lag == 0:
            t_years, v_years, te_years = 24, 8, 4
        elif lag == 5:
            t_years, v_years, te_years = 21, 6, 4
        else:
            raise ValueError("Only lag 0 and 5 are supported.")
    else:
        t_years, v_years, te_years = 18, 9, 3

    train_loader, val_loader, test_loader = get_dataloaders(
        batch_size=8, memory_years=5,
        train_years=t_years, val_years=v_years, test_years=te_years,
        data_type=data_type, lag=lag
    )

    model = CrabTransformer(
        grid_size=50, patch_size=5, in_channels=17,
        embed_dim=128, num_heads=8, num_layers=6, d_ff=512, dropout=0
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {n_params:,}")

    all_targets = []
    for inputs, tgts, temp_mask, y_idx, spat_mask, val_year in train_loader:
        for i in range(tgts.shape[0]):
            if val_year[i] > 0: # Skip 2020
                valid = tgts[i, 0][spat_mask[i] > 0]
                all_targets.append(valid.cpu().numpy())

    mean_log_target = np.concatenate(all_targets).mean()
    nn.init.constant_(model.decoder.conv_out.bias, mean_log_target)
    print(f"Decoder bias set to {mean_log_target:.3f}")

    # =======================================================
    # Grab ONE sample that is GUARANTEED to not be 2020
    # =======================================================
    for inputs_b, targets_b, temp_mask_b, y_idx_b, spat_mask_b, val_year_b in train_loader:
        valid_indices = torch.where(val_year_b > 0)[0]

        if len(valid_indices) > 0:
            idx = valid_indices[0]
            inputs = inputs_b[idx:idx+1].to(device)
            targets = targets_b[idx:idx+1].to(device)
            temporal_mask = temp_mask_b[idx:idx+1].to(device)
            year_idx = y_idx_b[idx:idx+1].to(device)
            spatial_mask = spat_mask_b[idx:idx+1].to(device)
            valid_year = val_year_b[idx:idx+1].to(device)
            break

    print(f"Selected Year Index {year_idx.item()} for overfitting test.")

    # ---> FIX: Ensure criterion is actually defined <---
    if loss_criterion == 'Tweedie':
        # Uncomment and use your custom Tweedie loss here
        # criterion = TweedieLoss(power=1.5)
        pass
        targets = torch.expm1(targets).clamp(min=1e-6)
    elif loss_criterion == 'MSE':
        # criterion = MSELoss_cm()
        criterion = nn.MSELoss()

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    print("\n🚀 Training for 1000 epochs on a single sample...")
    model.train()

    for epoch in range(1000):
        optimizer.zero_grad()

        output, _ = model(inputs, year_idx, temporal_mask, spatial_mask=spatial_mask, return_attention=True)

        if loss_criterion == 'Tweedie':
            output_loss = torch.expm1(output).clamp(min=1e-6)
            loss = criterion(output_loss, targets) # add mask=spatial_mask if your custom loss supports it
        elif loss_criterion == 'MSE':
            loss = criterion(output, targets)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        if (epoch + 1) % 100 == 0:
            print(f"   Epoch {epoch+1:4d} | Loss: {loss.item():.8f}")

    # Final verification
    model.eval()
    with+ torch.no_grad():
        final_output, _ = model(inputs, year_idx, temporal_mask, spatial_mask=spatial_mask, return_attention=True)

        valid_mask = spatial_mask.unsqueeze(1) > 0

        if loss_criterion == 'Tweedie':
            pred_np = torch.expm1(final_output)[valid_mask].cpu().numpy().flatten()
        else:
            pred_np = final_output[valid_mask].cpu().numpy().flatten()

        targ_np = targets[valid_mask].cpu().numpy().flatten()
        corr = np.corrcoef(pred_np, targ_np)[0, 1]

    print("\n" + "=" * 70)
    print("🏁 FINAL DIAGNOSTIC VERDICT:")
    print("=" * 70)

    if corr > 0.98:
        print(f"✅ SUCCESS! Final Correlation: {corr:.4f}")
    else:
        print(f"❌ FAILURE. Final Correlation: {corr:.4f}")

if __name__ == "__main__":
    test_overfit_on_real_data(data_type='real', loss_criterion='MSE', lag=0)
    test_overfit_on_real_data(data_type='real', loss_criterion='MSE', lag=5)

OVERFITTING TEST (real/real, MSE, LAG=0)
Loaded 100 bootstrap samples, 24 years each
Applying Log-Scaling to Spawners/Recruits ONLY.
Loaded 100 bootstrap samples, 8 years each
  Using 5 years of historical data from previous split
Applying Log-Scaling to Spawners/Recruits ONLY.
Loaded 100 bootstrap samples, 4 years each
  Using 5 years of historical data from previous split
Applying Log-Scaling to Spawners/Recruits ONLY.
Sinusoidal temporal encoding initialized:
  Pre-computed years: 0-49
  Can dynamically compute beyond year 50 ✓
Initializing model weights...
✓ Applying Bias Initialization Surgery to Decoder (-1.0)
✓ Weight initialization complete
Trainable parameters: 1,321,249
Decoder bias set to 2.819
Selected Year Index 10 for overfitting test.

🚀 Training for 1000 epochs on a single sample...
   Epoch  100 | Loss: 0.11154789
   Epoch  200 | Loss: 0.02331997
   Epoch  300 | Loss: 0.00999909
   Epoch  400 | Loss: 0.00742243
   Epoch  500 | Loss: 0.00535685
   Epoch  600 | Loss: 0.0

In [10]:
"""
run_batch_evaluation.py  —  Evaluate all trained CrabTransformer checkpoints.

Auto-discovers checkpoints by scanning for best_model.pt under DRIVE_BASE.
Parses run config entirely from the directory path (no manual config list).

For every sample, metrics are computed TWICE:
  raw_*    — back-transformed predictions, no additional thresholding
  thresh_* — both pred and target zeroed below ZERO_THRESHOLD (14.23)
             before all metric calculations

Outputs (saved to SAVE_DIR)
----------------------------
  full_results.csv     one row per (model_size/level/channel_cfg/pred_mode/
                       criterion/phase/year/bootstrap)
  summary_results.csv  mean ± SD per (config × phase)
  plots/{run_id}/
    abundance.png      median pred vs obs trajectory with bootstrap IQR
    spatial_grids.png  representative spatial maps for train / val / test
"""

import os
import sys
import json
import glob
import traceback

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr

# ---- adjust to your layout ----
REPO_DIR   = '/content/Teleconnections-ViT'
DRIVE_BASE = '/content/drive/MyDrive/Teleconnection_ViT/model_outputs'
SAVE_DIR   = '/content/drive/MyDrive/Teleconnection_ViT/analysis'
# --------------------------------

sys.path.insert(0, REPO_DIR)
from models.model     import CrabTransformer
from data.data_helper import get_dataloaders

ZERO_THRESHOLD  = 14.23
TOP_K_FRACTION  = 0.10
BATCH_SIZE      = 8
MEMORY_YEARS    = 5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# ============================================================
#  YEAR SPLITS (must match train.py)
# ============================================================

def get_year_splits(data_type, lag):
    if data_type == 'real':
        return (24, 8, 4) if lag == 0 else (21, 6, 4)
    return (18, 9, 3)   # dummy data — no lag5 splits exist


# ============================================================
#  METRICS
# ============================================================

def compute_metrics(p_flat: np.ndarray, t_flat: np.ndarray,
                    train_mean: float = None) -> dict:
    """
    Full metric suite for one spatial sample (1-D arrays of valid cells).

    Returns a flat dict.  Returns an empty dict for degenerate inputs.
    """
    n = len(p_flat)
    if n == 0:
        return {}

    # Spatial correlation
    if np.std(p_flat) > 0 and np.std(t_flat) > 0:
        spear, _ = spearmanr(p_flat, t_flat)
        pear,  _ = pearsonr(p_flat,  t_flat)
    else:
        spear = pear = 0.0

    # Error decomposition
    diff          = p_flat - t_flat
    mae           = float(np.mean(np.abs(diff)))
    rmse          = float(np.sqrt(np.mean(diff ** 2)))
    bias          = float(np.mean(diff))
    unbiased_rmse = float(np.sqrt(max(rmse ** 2 - bias ** 2, 0.0)))
    bias_frac_pct = float(bias ** 2 / (rmse ** 2 + 1e-10) * 100)

    # Abundance
    pred_total        = float(p_flat.sum())
    obs_total         = float(t_flat.sum())
    abundance_capture = pred_total / (obs_total + 1e-6)
    pct_abund_error   = (pred_total - obs_total) / (obs_total + 1e-6) * 100

    # Top-10 % Jaccard
    k            = max(1, int(n * TOP_K_FRACTION))
    pred_top     = set(np.argsort(p_flat)[-k:])
    obs_top      = set(np.argsort(t_flat)[-k:])
    intersect    = len(pred_top & obs_top)
    union        = len(pred_top | obs_top)
    jaccard      = intersect / union if union > 0 else 0.0

    # Zero-cell classification (threshold = 14.23 applied BEFORE calling this
    # function for the thresh variant; the raw variant has no prior zeroing)
    pred_zero    = p_flat < ZERO_THRESHOLD
    obs_zero     = t_flat < ZERO_THRESHOLD
    tp = float(np.sum( pred_zero &  obs_zero))
    fp = float(np.sum( pred_zero & ~obs_zero))
    fn = float(np.sum(~pred_zero &  obs_zero))
    precision    = tp / (tp + fp + 1e-10)
    recall       = tp / (tp + fn + 1e-10)
    f1           = 2 * precision * recall / (precision + recall + 1e-10)

    # Skill scores vs climatological mean baseline
    if train_mean is not None:
        base_mse  = float(np.mean((t_flat - train_mean) ** 2))
        base_mae  = float(np.mean(np.abs(t_flat - train_mean)))
        skill_mse = float(1.0 - np.mean(diff ** 2) / (base_mse + 1e-10))
        skill_mae = float(1.0 - mae               / (base_mae  + 1e-10))
    else:
        skill_mse = skill_mae = float('nan')

    return {
        'spearman':          float(spear),
        'pearson':           float(pear),
        'mae':               mae,
        'rmse':              rmse,
        'bias':              bias,
        'unbiased_rmse':     unbiased_rmse,
        'bias_frac_pct':     bias_frac_pct,
        'pred_total':        pred_total,
        'obs_total':         obs_total,
        'abundance_capture': abundance_capture,
        'pct_abund_error':   float(pct_abund_error),
        'top10_jaccard':     float(jaccard),
        'zero_precision':    float(precision),
        'zero_recall':       float(recall),
        'zero_f1':           float(f1),
        'skill_mse':         skill_mse,
        'skill_mae':         skill_mae,
    }


# ============================================================
#  PLOTS
# ============================================================

def save_abundance_plot(records, run_id, plot_dir, train_end, val_end, n_years):
    df  = pd.DataFrame(records)
    ann = df.groupby('year').agg(
        pred_median=('raw_pred_total', 'median'),
        pred_p25   =('raw_pred_total', lambda x: np.percentile(x, 25)),
        pred_p75   =('raw_pred_total', lambda x: np.percentile(x, 75)),
        obs_median =('raw_obs_total',  'median'),
    ).reset_index()

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.fill_between(ann['year'], ann['pred_p25'], ann['pred_p75'],
                    alpha=0.25, color='steelblue', label='Pred IQR')
    ax.plot(ann['year'], ann['pred_median'], 'b-o', ms=4, lw=2, label='Pred median')
    ax.plot(ann['year'], ann['obs_median'],  'k-o', ms=4, lw=2, label='Obs median')
    ax.axvline(train_end - 0.5, color='grey', ls=':', lw=1.5)
    ax.axvline(val_end   - 0.5, color='grey', ls=':', lw=1.5)
    t = ax.get_xaxis_transform()
    ax.text(train_end / 2,               0.97, 'TRAIN', transform=t,
            ha='center', color='seagreen',   fontsize=9, fontweight='bold')
    ax.text((train_end + val_end) / 2,   0.97, 'VAL',   transform=t,
            ha='center', color='darkorange', fontsize=9, fontweight='bold')
    ax.text((val_end + n_years) / 2,     0.97, 'TEST',  transform=t,
            ha='center', color='crimson',    fontsize=9, fontweight='bold')
    ax.set_xlabel('Year index'); ax.set_ylabel('Total abundance')
    ax.set_title(f'Abundance — {run_id}', fontsize=9)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, 'abundance.png'), dpi=150, bbox_inches='tight')
    plt.close()


def save_spatial_grid(records, run_id, plot_dir, train_end, val_end):
    phase_target = {'TRAIN': train_end // 2,
                    'VAL':   train_end + 2,
                    'TEST':  val_end   + 1}
    phase_color  = {'TRAIN': 'seagreen', 'VAL': 'darkorange', 'TEST': 'crimson'}

    fig, axes = plt.subplots(3, 3, figsize=(11, 11))
    titles    = ['Spawner / ch-0 (log)', 'True Recruits (log)', 'Pred Recruits (log)']

    for row, (phase, tgt_yr) in enumerate(phase_target.items()):
        cands = sorted([r for r in records if r['phase'] == phase],
                       key=lambda r: abs(r['year'] - tgt_yr))
        if not cands:
            for col in range(3):
                axes[row, col].axis('off')
            continue
        r   = cands[0]
        col_0_img = r.get('input_ch0_log', np.zeros((50, 50)))
        imgs  = [col_0_img, r['target_log'], r['pred_log']]
        for col, (img, ttl) in enumerate(zip(imgs, titles)):
            ax = axes[row, col]
            ax.imshow(img, cmap='viridis' if col == 0 else 'plasma',
                      vmin=0, vmax=8, interpolation='nearest')
            if row == 0:
                ax.set_title(ttl, fontsize=9)
            ax.set_ylabel(f'Yr {r["year"]} [{phase}]' if col == 0 else '')
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_edgecolor(phase_color[phase])
                sp.set_linewidth(2.5); sp.set_visible(True)

    plt.suptitle(f'Spatial grids — {run_id}', fontsize=10, y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, 'spatial_grids.png'), dpi=150, bbox_inches='tight')
    plt.close()


# ============================================================
#  EVALUATE ONE RUN
# ============================================================

def evaluate_run(run_dir: str) -> list:
    """
    Load checkpoint, run inference on all three splits, return per-sample records.
    Config is recovered from the directory path + training_history.json.
    """
    # -- Parse path ---
    rel   = os.path.relpath(run_dir, DRIVE_BASE)
    parts = rel.split(os.sep)
    if len(parts) != 5:
        raise ValueError(f"Unexpected depth: {rel}  (need 5 levels)")
    model_size, level, channel_cfg, pred_mode, criterion = parts
    data_type = 'real' if level == 'real' else 'dummy'
    run_id    = rel.replace(os.sep, '/')

    # -- Load training history (contains channel metadata) ---
    history_path = os.path.join(run_dir, 'training_history.json')
    if not os.path.exists(history_path):
        raise FileNotFoundError(f"No training_history.json in {run_dir}")
    with open(history_path) as f:
        hist = json.load(f)

    bias_correction = hist.get('bias_correction', 1.0)
    meta            = hist.get('channel_cfg_meta', {})

    # Recover channel metadata from JSON (saved by train.py)
    in_channels          = meta['in_channels']
    channel_mask_indices = meta['channel_mask_indices']
    use_spawners         = meta['use_spawners']
    use_recruits         = meta['use_recruits']
    use_temp             = meta['use_temp']
    incl_curr            = meta['incl_curr']
    lag                  = meta['lag']
    embed_dim            = meta.get('embed_dim', 128)
    num_heads            = meta.get('num_heads', 8)
    num_layers           = meta.get('num_layers', 6)
    d_ff                 = meta.get('d_ff', 512)

    t_years, v_years, te_years = get_year_splits(data_type, lag)

    # -- Model ---
    model = CrabTransformer(
        grid_size=50, patch_size=5,
        in_channels=in_channels,
        embed_dim=embed_dim,
        num_heads=num_heads,
        num_layers=num_layers,
        d_ff=d_ff,
        dropout=0.0,
        channel_mask_indices=channel_mask_indices,
    ).to(device)
    model.load_state_dict(
        torch.load(os.path.join(run_dir, 'best_model.pt'), map_location=device)
    )
    model.eval()

    # -- Data ---
    train_loader, val_loader, test_loader = get_dataloaders(
        batch_size=BATCH_SIZE, memory_years=MEMORY_YEARS,
        train_years=t_years, val_years=v_years, test_years=te_years,
        level=level, data_type=data_type,
        include_current_spawner=incl_curr,
        lag=lag,
        use_temp=use_temp,
        use_spawners=use_spawners,
        use_recruits=use_recruits,
    )

    # Spatial validity mask
    if data_type == 'real':
        valid_mask = np.load('data/real/output/spatial_mask.npy') > 0
    else:
        valid_mask = np.ones((50, 50), dtype=bool)

    # Training mean for skill scores
    train_vals = []
    with torch.no_grad():
        for _, targets, _, _, spatial_mask, valid_year in train_loader:
            for i in range(targets.shape[0]):
                if valid_year[i] > 0:
                    train_vals.append(np.expm1(targets[i, 0].numpy())[valid_mask])
    train_mean = float(np.concatenate(train_vals).mean()) if train_vals else None

    n_years_total = t_years + v_years + te_years
    loaders = [
        ('TRAIN', train_loader),
        ('VAL',   val_loader),
        ('TEST',  test_loader),
    ]

    records = []
    seen_years_in_plot = {}          # phase → set of years already stored for plots

    with torch.no_grad():
        for phase, loader in loaders:
            seen_years_in_plot[phase] = set()

            for batch in loader:
                inputs, targets, temporal_mask, year_idx, spatial_mask, valid_year = batch

                inputs        = inputs.to(device)
                temporal_mask = temporal_mask.to(device)
                year_idx_dev  = year_idx.to(device)
                spatial_mask  = spatial_mask.to(device)

                outputs = model(inputs, year_idx_dev, temporal_mask,
                                spatial_mask=spatial_mask)

                p_log = outputs.cpu().numpy()
                t_log = targets.numpy()
                y_abs = year_idx.numpy()
                vy    = valid_year.numpy()

                for i in range(p_log.shape[0]):
                    if vy[i] == 0:
                        continue

                    yr = int(y_abs[i])

                    # Back-transform: exp(log1p_pred) × bias_correction - 1
                    p_raw = np.clip(np.exp(p_log[i, 0]) * bias_correction - 1.0,
                                    0, None)
                    t_raw = np.expm1(t_log[i, 0])
                    p_raw[~valid_mask] = 0.0
                    t_raw[~valid_mask] = 0.0

                    p_flat = p_raw[valid_mask]
                    t_flat = t_raw[valid_mask]

                    # ---- RAW metrics ----
                    raw_m = compute_metrics(p_flat, t_flat, train_mean)

                    # ---- THRESHOLDED metrics ----
                    p_thr = p_flat.copy();  p_thr[p_thr < ZERO_THRESHOLD] = 0.0
                    t_thr = t_flat.copy();  t_thr[t_thr < ZERO_THRESHOLD] = 0.0
                    thr_m = compute_metrics(p_thr, t_thr, train_mean)

                    rec = {
                        'model_size':  model_size,
                        'level':       level,
                        'channel_cfg': channel_cfg,
                        'pred_mode':   pred_mode,
                        'criterion':   criterion,
                        'phase':       phase,
                        'year':        yr,
                        'bias_correction': bias_correction,
                        'raw_pred_total': float(p_flat.sum()),
                        'raw_obs_total':  float(t_flat.sum()),
                    }
                    for k, v in raw_m.items():
                        rec[f'raw_{k}'] = v
                    for k, v in thr_m.items():
                        rec[f'thresh_{k}'] = v

                    ## Store grids for the first bootstrap per year (for plots)
                    #if yr not in seen_years_in_plot[phase]:
                    #    seen_years_in_plot[phase].add(yr)
                    #    # channel 0 is first input channel if spawners present,
                    #    # otherwise recruits t-1; just show input[:,0] as proxy
                    #    rec['input_ch0_log'] = inputs[i, 0].cpu().numpy()
                    #    rec['target_log']    = t_log[i, 0]
                    #    rec['pred_log']      = p_log[i, 0]

                    records.append(rec)

    # -- Per-run plots --
    #plot_dir = os.path.join(SAVE_DIR, 'plots', run_id.replace('/', '_'))
    #os.makedirs(plot_dir, exist_ok=True)
    #try:
    #    # Filter to records that have grid arrays
    #    plot_recs = [r for r in records if 'target_log' in r]
    #    save_abundance_plot(records, run_id, plot_dir,
    #                        t_years, t_years + v_years, n_years_total)
    #    save_spatial_grid(plot_recs, run_id, plot_dir,
    #                      t_years, t_years + v_years)
    #except Exception as e:
    #    print(f"  ⚠  Plot failed: {e}")

    # Drop large array fields before returning (keep CSVs lean)
    for r in records:
        r.pop('input_ch0_log', None)
        r.pop('target_log',    None)
        r.pop('pred_log',      None)

    return records


# ============================================================
#  SUMMARY
# ============================================================

def make_summary(df: pd.DataFrame) -> pd.DataFrame:
    group_cols  = ['model_size', 'level', 'channel_cfg', 'pred_mode', 'criterion', 'phase']
    metric_cols = [c for c in df.columns if c.startswith('raw_') or c.startswith('thresh_')]
    means  = df.groupby(group_cols)[metric_cols].mean().add_suffix('_mean')
    stds   = df.groupby(group_cols)[metric_cols].std().add_suffix('_std')
    counts = df.groupby(group_cols)[metric_cols[0]].count().rename('n_samples')
    return pd.concat([means, stds, counts], axis=1).reset_index().round(4)


def print_overview(df: pd.DataFrame):
    test = df[df['phase'] == 'TEST']
    if test.empty:
        print("No TEST records."); return
    agg = (test.groupby(['model_size', 'level', 'channel_cfg', 'pred_mode', 'criterion'])
               .agg(spearman=('raw_spearman',         'mean'),
                    spear_sd =('raw_spearman',         'std'),
                    abund    =('raw_abundance_capture','mean'),
                    skill    =('raw_skill_mse',        'mean'),
                    n        =('raw_spearman',         'count'))
               .reset_index()
               .sort_values('spearman', ascending=False))
    pd.set_option('display.float_format', '{:.3f}'.format)
    pd.set_option('display.max_rows', 300)
    print("\n=== TEST SET OVERVIEW (raw Spearman, sorted desc) ===")
    print(agg.to_string(index=False))
    pd.reset_option('display.float_format')


# ============================================================
#  MAIN
# ============================================================

def run_all_evaluations():
    os.makedirs(SAVE_DIR, exist_ok=True)

    ckpt_paths = sorted(glob.glob(
        os.path.join(DRIVE_BASE, '*', '*', '*', '*', '*', 'best_model.pt')
    ))
    run_dirs = [os.path.dirname(p) for p in ckpt_paths]

    print(f"\n{'='*72}")
    print(f"  CrabTransformer — Batch Evaluation")
    print(f"  Found {len(run_dirs)} trained models under {DRIVE_BASE}")
    print(f"  Saving to {SAVE_DIR}")
    print(f"{'='*72}\n")

    all_records = []
    failed      = []

    for i, run_dir in enumerate(run_dirs, 1):
        rel = os.path.relpath(run_dir, DRIVE_BASE)
        print(f"[{i:3d}/{len(run_dirs)}]  {rel}")
        try:
            recs = evaluate_run(run_dir)
            all_records.extend(recs)
            print(f"  ✅  {len(recs)} samples")
        except Exception as e:
            print(f"  ❌  {e}")
            traceback.print_exc()
            failed.append({'index': i, 'dir': rel, 'error': str(e)})

    if not all_records:
        print("No records collected — nothing to save.")
        return

    df = pd.DataFrame(all_records)

    full_path = os.path.join(SAVE_DIR, 'full_results.csv')
    df.to_csv(full_path, index=False)
    print(f"\n✅  Full results  → {full_path}  ({len(df):,} rows)")

    summary = make_summary(df)
    summ_path = os.path.join(SAVE_DIR, 'summary_results.csv')
    summary.to_csv(summ_path, index=False)
    print(f"✅  Summary       → {summ_path}  ({len(summary):,} rows)")

    print_overview(df)

    if failed:
        print(f"\n⚠  {len(failed)} runs failed:")
        for f in failed:
            print(f"  [{f['index']}] {f['dir']}  →  {f['error']}")

    print(f"\n{'='*72}")
    print(f"Evaluation complete. "
          f"{len(run_dirs) - len(failed)}/{len(run_dirs)} runs succeeded.")
    print(f"{'='*72}\n")


if __name__ == '__main__':
    run_all_evaluations()


  CrabTransformer — Batch Evaluation
  Found 144 trained models under /content/drive/MyDrive/Teleconnection_ViT/model_outputs
  Saving to /content/drive/MyDrive/Teleconnection_ViT/analysis

[  1/144]  normal/easy/recruits_only/normal/MSE
TemporalEncoding: pre-computed 0–49, dynamic fallback beyond that.
Initialising model weights …
  Decoder conv_out bias initialised to 0.0 (will be overridden by warm-start in train.py)
  Weight initialisation complete.
Loaded 100 bootstraps × 18 years | spawners=False  recruits=True  temp=False
Applying log1p scaling to spawners / recruits.
  in_channels=5  channel_mask_indices=[1, 2, 3, 4, 5]
Loaded 100 bootstraps × 9 years | spawners=False  recruits=True  temp=False
  Using 5 years of historical data from previous split
Applying log1p scaling to spawners / recruits.
  in_channels=5  channel_mask_indices=[1, 2, 3, 4, 5]
Loaded 100 bootstraps × 3 years | spawners=False  recruits=True  temp=False
  Using 5 years of historical data from previous split


In [ ]:
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')  # Fix Agg
import matplotlib.pyplot as plt
import torch
import numpy as np
import os
import json
from data.data_helper import get_dataloaders

DRIVE_DIR = "/content/drive/MyDrive/Teleconnection_ViT"
SAVE_DIR = os.path.join(DRIVE_DIR, "analysis/batch_evaluation")
os.makedirs(SAVE_DIR, exist_ok=True)

def generate_comparison_panel(levels=['easy', 'medium', 'hard'], num_reps=100,
                              data_type='dummy'):
    n_panels = len(levels)
    fig, axes = plt.subplots(1, n_panels, figsize=(8 * n_panels, 8), sharey=True)
    if n_panels == 1:
        axes = [axes]

    TRAIN_YEARS = 22 if data_type == 'real' else 22
    VAL_YEARS = 9
    TEST_YEARS = 3
    N_YEARS = TRAIN_YEARS + VAL_YEARS + TEST_YEARS
    TRAIN_END = TRAIN_YEARS
    VAL_END = TRAIN_YEARS + VAL_YEARS

    # Load spatial mask for real data
    if data_type == 'real':
        valid_mask = np.load("data/real/output/spatial_mask.npy") > 0  # [50, 50]
    else:
        valid_mask = np.ones((50, 50), dtype=bool)

    def add_phase_regions(ax):
        ax.axvline(TRAIN_END - 0.5, color="grey", ls=":", lw=1.2, alpha=0.8)
        ax.axvline(VAL_END - 0.5, color="grey", ls=":", lw=1.2, alpha=0.8)
        ax.axvspan(-0.5, TRAIN_END - 0.5, color='green', alpha=0.03)
        ax.axvspan(TRAIN_END - 0.5, VAL_END - 0.5, color='orange', alpha=0.03)
        ax.axvspan(VAL_END - 0.5, N_YEARS - 0.5, color='red', alpha=0.03)

    for col, level in enumerate(levels):
        eval_level = 'real' if data_type == 'real' else level

        train_loader, val_loader, test_loader = get_dataloaders(
            batch_size=1, level=eval_level, transform='log',
            train_years=TRAIN_YEARS, val_years=VAL_YEARS, test_years=TEST_YEARS,
            data_type=data_type
        )

        full_truth, full_spawners = [], []

        with torch.no_grad():
            for loader in [train_loader, val_loader, test_loader]:
                for batch in loader:
                    inputs = batch[0]
                    targets = batch[1]

                    # Sum only over valid ocean cells
                    spawner_raw = torch.expm1(inputs[:, 0]).cpu().numpy()
                    recruit_raw = torch.expm1(targets[:, 0]).cpu().numpy()

                    full_spawners.append(
                        spawner_raw[:, valid_mask].sum(axis=1).flatten()
                    )
                    full_truth.append(
                        recruit_raw[:, valid_mask].sum(axis=1).flatten()
                    )

        def process_reps(data_list):
            arr = np.concatenate(data_list).reshape(num_reps, N_YEARS)
            return (
                np.percentile(arr, 50, axis=0),
                np.percentile(arr, 5, axis=0),
                np.percentile(arr, 95, axis=0),
            )

        mid_truth, lo_truth, hi_truth = process_reps(full_truth)
        mid_spwn,  lo_spwn,  hi_spwn  = process_reps(full_spawners)

        time_x = np.arange(N_YEARS)
        ax = axes[col]
        add_phase_regions(ax)

        ax.plot(time_x, mid_spwn, color='green', label='Spawners (median)',
                lw=2.5, ls='--')
        ax.fill_between(time_x, lo_spwn, hi_spwn, color='green', alpha=0.08)

        ax.plot(time_x, mid_truth, color='black', label='Recruits (median)',
                lw=2, zorder=10)
        ax.fill_between(time_x, lo_truth, hi_truth, color='black', alpha=0.08)

        ax.set_title(f"SCENARIO: {eval_level.upper()}", fontweight='bold')
        ax.set_xlabel("Year")
        if col == 0:
            ax.set_ylabel("Total Abundance")
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f"data_comparison_panel_{data_type}.png"))
    plt.show()

# Dummy data
# generate_comparison_panel(levels=['easy', 'medium', 'hard'], data_type='dummy')

# Real data
generate_comparison_panel(levels=['real'], num_reps=100, data_type='real')

Plot for Maia

In [ ]:
"""
=============================================================================
Integrated Gradients for CrabTransformer (Targeted Runs)
=============================================================================
Post-hoc attribution analysis. Run AFTER training is complete.
"""

import os
import json
import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt

# Adjust based on your environment
REPO_DIR   = '/content/Teleconnections-ViT'
import sys
sys.path.insert(0, REPO_DIR)

from models.model import CrabTransformer
from data.data_helper import get_dataloaders

# =============================================================================
# CONFIGURATION
# =============================================================================

DRIVE_DIR  = '/content/drive/MyDrive/Teleconnection_ViT'
OUTPUTS_DIR = os.path.join(DRIVE_DIR, 'model_outputs')
SAVE_DIR   = os.path.join(DRIVE_DIR, 'analysis/attribution')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CHANNEL_NAMES = [
    'Spawner (t)', 'Spawner (t-1)', 'Spawner (t-2)', 'Spawner (t-3)', 'Spawner (t-4)', 'Spawner (t-5)',
    'Recruit (t-1)', 'Recruit (t-2)', 'Recruit (t-3)', 'Recruit (t-4)', 'Recruit (t-5)',
    'Temp (t)', 'Temp (t-1)', 'Temp (t-2)', 'Temp (t-3)', 'Temp (t-4)', 'Temp (t-5)'
]

os.makedirs(SAVE_DIR, exist_ok=True)

# =============================================================================
# STEP 1: COMPUTE BASELINE (mean field from training data)
# =============================================================================

def compute_baseline(train_loader):
    print('  Computing baseline (mean training field)...')
    all_inputs = []
    for inputs, _, _, _, _, _ in train_loader:
        all_inputs.append(inputs)

    stacked = torch.cat(all_inputs, dim=0)
    baseline = stacked.mean(dim=0, keepdim=True)

    print(f'    Baseline from {stacked.shape[0]} samples')
    print(f'    Range: [{baseline.min():.3f}, {baseline.max():.3f}]')
    return baseline


# =============================================================================
# STEP 2: ORGANIZE SAMPLES BY YEAR
# =============================================================================

def collect_samples_by_year(loader):
    by_year = {}

    for inputs, targets, mask, year_idx, spat_mask, val_year in loader:
        batch_size = inputs.shape[0]
        for i in range(batch_size):
            if val_year[i] == 0:
                continue

            yr = int(year_idx[i].item())
            if yr not in by_year:
                by_year[yr] = []
            by_year[yr].append((
                inputs[i:i+1],
                targets[i:i+1],
                mask[i:i+1],
                year_idx[i:i+1],
            ))

    for yr in sorted(by_year.keys()):
        print(f'    Year {yr}: {len(by_year[yr])} bootstrap samples')

    return by_year


# =============================================================================
# STEP 3: INTEGRATED GRADIENTS (core computation)
# =============================================================================

def integrated_gradients(model, actual_input, baseline, year_idx,
                         temporal_mask, n_steps=50, output_fn=None):
    if output_fn is None:
        output_fn = lambda out: out.mean()

    delta = actual_input - baseline
    accumulated_grads = torch.zeros_like(actual_input)

    for step in range(n_steps):
        alpha = step / n_steps
        interpolated = baseline + alpha * delta
        interpolated = interpolated.detach().clone().requires_grad_(True)

        output = model(interpolated, year_idx, temporal_mask)
        scalar_output = output_fn(output)

        model.zero_grad()
        scalar_output.backward()

        accumulated_grads += interpolated.grad.detach()

    attribution = (accumulated_grads / n_steps) * delta

    return attribution.squeeze(0).cpu().numpy()


# =============================================================================
# STEP 4: RUN IG PER YEAR, AVERAGE ACROSS BOOTSTRAPS
# =============================================================================

def compute_yearly_attributions(model, by_year, baseline, n_steps=50, output_fn=None):
    baseline = baseline.to(DEVICE)
    yearly_attr = {}
    yearly_inputs = {}

    for yr in sorted(by_year.keys()):
        samples = by_year[yr]
        n_bootstraps = len(samples)
        print(f'\n  Year {yr}: running IG on {n_bootstraps} bootstraps...')

        attr_list = []
        input_list = []

        for idx, (inp, tgt, msk, yr_idx) in enumerate(samples):
            inp_dev  = inp.to(DEVICE)
            yr_dev   = yr_idx.to(DEVICE)
            msk_dev  = msk.to(DEVICE)

            attr = integrated_gradients(
                model, inp_dev, baseline,
                yr_dev, msk_dev,
                n_steps=n_steps, output_fn=output_fn
            )
            attr_list.append(attr)
            input_list.append(inp.squeeze(0).numpy())

            if (idx + 1) % 25 == 0:
                print(f'    {idx + 1}/{n_bootstraps} done')

        yearly_attr[yr]   = np.stack(attr_list).mean(axis=0)
        yearly_inputs[yr] = np.stack(input_list).mean(axis=0)

        print(f'    Year {yr} done. Attr range: [{yearly_attr[yr].min():.6f}, {yearly_attr[yr].max():.6f}]')

    return yearly_attr, yearly_inputs


# =============================================================================
# STEP 5: VISUALIZATION
# =============================================================================

def plot_year_panel(yr, mean_input, mean_attr, meta, out_name):
    use_temp = meta['use_temp']
    n_channels = meta['in_channels']

    rows_per_half = 3 if use_temp else 2
    total_rows = rows_per_half * 2

    fig, axes = plt.subplots(total_rows, 6, figsize=(24, 4 * total_rows))

    abs_attr = np.abs(mean_attr)
    vmax_attr = np.percentile(abs_attr, 99)

    # ── Top Half: Mean Input Channels ──
    for c in range(n_channels):
        row, col = c // 6, c % 6
        ax = axes[row, col]
        ax.imshow(mean_input[c], cmap='viridis', vmin=0, vmax=8 if c < 11 else None)
        ax.set_title(f'Input: {CHANNEL_NAMES[c]}', fontsize=8, fontweight='bold')
        ax.axis('off')

    # Aggregate input spot
    ax = axes[n_channels // 6, n_channels % 6]
    input_agg = np.abs(mean_input).sum(axis=0)
    ax.imshow(input_agg, cmap='viridis')
    ax.set_title('Input: Aggregate', fontsize=8, fontweight='bold')
    ax.axis('off')

    # ── Bottom Half: Attribution Channels ──
    for c in range(n_channels):
        row, col = rows_per_half + (c // 6), c % 6
        ax = axes[row, col]
        im = ax.imshow(mean_attr[c], cmap='RdBu_r', vmin=-vmax_attr, vmax=vmax_attr)
        ax.set_title(f'Attr: {CHANNEL_NAMES[c]}', fontsize=8, fontweight='bold')
        ax.axis('off')

    # Aggregate |attribution| spot
    ax = axes[rows_per_half + (n_channels // 6), n_channels % 6]
    agg_attr = abs_attr.sum(axis=0)
    im_agg = ax.imshow(agg_attr, cmap='hot')
    ax.set_title('Attr: Aggregate |IG|', fontsize=8, fontweight='bold')
    ax.axis('off')

    fig.suptitle(
        f'Year {yr} — {out_name}\n'
        f'Top: Mean Input   |   Bottom: Attribution (red=+recruit, blue=-recruit)',
        fontsize=16, fontweight='bold'
    )
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    path = os.path.join(SAVE_DIR, f'ig_year{yr}_{out_name}.png')
    plt.savefig(path, dpi=150)
    print(f'  Saved: {path}')
    plt.close()


def plot_grand_average(yearly_attr, yearly_inputs, meta, out_name):
    years = sorted(yearly_attr.keys())
    grand_attr  = np.stack([yearly_attr[yr] for yr in years]).mean(axis=0)
    grand_input = np.stack([yearly_inputs[yr] for yr in years]).mean(axis=0)

    plot_year_panel('AVG', grand_input, grand_attr, meta, out_name)


def plot_channel_bar_yearly(yearly_attr, meta, out_name):
    years = sorted(yearly_attr.keys())

    data = {}
    for yr in years:
        totals = np.abs(yearly_attr[yr]).sum(axis=(1, 2))
        data[f'Yr {yr}'] = totals / totals.sum() * 100

    grand = np.stack([yearly_attr[yr] for yr in years]).mean(axis=0)
    grand_totals = np.abs(grand).sum(axis=(1, 2))
    data['Average'] = grand_totals / grand_totals.sum() * 100

    # Read all flags
    use_temp = meta['use_temp']
    use_recruits = meta['use_recruits']
    incl_curr = meta['incl_curr']
    n_channels = meta['in_channels']

    # --- THE FIX: Conditionally build the entire active names list ---
    active_names = []

    # 1. Spawner names (always present in these 3 models)
    if incl_curr: active_names.append('Spawner (t)')
    active_names.extend(['Spawner (t-1)', 'Spawner (t-2)', 'Spawner (t-3)', 'Spawner (t-4)', 'Spawner (t-5)'])

    # 2. Recruit names
    if use_recruits:
        active_names.extend(['Recruit (t-1)', 'Recruit (t-2)', 'Recruit (t-3)', 'Recruit (t-4)', 'Recruit (t-5)'])

    # 3. Temp names
    if use_temp:
        if incl_curr: active_names.append('Temp (t)')
        active_names.extend(['Temp (t-1)', 'Temp (t-2)', 'Temp (t-3)', 'Temp (t-4)', 'Temp (t-5)'])
    # ----------------------------------------------------------------

    x = np.arange(n_channels)
    n_groups = len(data)
    width = 0.8 / n_groups
    colors = plt.cm.tab10(np.linspace(0, 1, n_groups))

    fig, ax = plt.subplots(figsize=(16, 6))

    for i, (label, pcts) in enumerate(data.items()):
        offset = (i - n_groups / 2 + 0.5) * width
        bars = ax.bar(x + offset, pcts, width, label=label,
                      color=colors[i], alpha=0.8, edgecolor='black',
                      linewidth=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(active_names, rotation=45, ha='right')
    ax.set_ylabel('Share of Total Attribution (%)')
    ax.set_title(
        f'Channel Attribution by Year: {out_name}',
        fontsize=14, fontweight='bold'
    )
    ax.legend(loc='upper right')
    ax.grid(axis='y', alpha=0.3, ls='--')
    plt.tight_layout()

    path = os.path.join(SAVE_DIR, f'ig_bar_yearly_{out_name}.png')
    plt.savefig(path, dpi=150)
    print(f'  Saved: {path}')
    plt.close()


# =============================================================================
# MAIN ENTRY POINT
# =============================================================================

def run_targeted_ig(model_size, level, channel_cfg, pred_mode, criterion, phase='TEST', n_steps=50):

    out_name = f"{model_size}_{level}_{channel_cfg}_{pred_mode}_{criterion}"

    print(f'\n{"="*70}')
    print(f' Running IG for: {out_name}')
    print(f' Phase: {phase} | Steps: {n_steps}')
    print(f'{"="*70}')

    # 1. Resolve exact directory matching train.py output structure
    ckpt_dir = os.path.join(OUTPUTS_DIR, model_size, level, channel_cfg, pred_mode, criterion)
    ckpt_path = os.path.join(ckpt_dir, 'best_model.pt')
    history_path = os.path.join(ckpt_dir, 'training_history.json')

    if not os.path.exists(ckpt_path) or not os.path.exists(history_path):
        print(f'  ⚠️ Missing model or history file in: {ckpt_dir}\n  Skipping...')
        return

    # 2. Load architectural metadata dynamically saved during training!
    with open(history_path, 'r') as f:
        meta = json.load(f)['channel_cfg_meta']

    print(f"  Loaded Meta -> Channels: {meta['in_channels']}, Lag: {meta['lag']}, Temp: {meta['use_temp']}")

    # 3. Load Model
    model = CrabTransformer(
        grid_size=50, patch_size=5,
        in_channels=meta['in_channels'],
        embed_dim=meta['embed_dim'],
        num_heads=meta['num_heads'],
        num_layers=meta['num_layers'],
        d_ff=meta['d_ff'],
        dropout=0,
        channel_mask_indices=meta['channel_mask_indices'] # Required for CrabTransformer!
    ).to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()

    # 4. Load Data (Always loads real data structure per these targeted runs)
    t_years, v_years, te_years = (24, 8, 4) if meta['lag'] == 0 else (21, 6, 4)

    train_loader, val_loader, test_loader = get_dataloaders(
        batch_size=8, memory_years=5,
        train_years=t_years, val_years=v_years, test_years=te_years,
        data_type='real', level=level,
        include_current_spawner=meta['incl_curr'],
        lag=meta['lag'],
        use_temp=meta['use_temp'],
        use_spawners=meta['use_spawners'],
        use_recruits=meta['use_recruits']
    )

    baseline = compute_baseline(train_loader)

    if phase == 'TEST':
        target_loader = test_loader
    elif phase == 'VAL':
        target_loader = val_loader
    else:
        target_loader = train_loader

    print(f'\n  Collecting {phase} samples by year...')
    by_year = collect_samples_by_year(target_loader)

    yearly_attr, yearly_inputs = compute_yearly_attributions(
        model, by_year, baseline, n_steps=n_steps
    )

    # ── Save raw arrays ──
    for yr in sorted(yearly_attr.keys()):
        np.save(os.path.join(SAVE_DIR, f'ig_attr_yr{yr}_{out_name}.npy'), yearly_attr[yr])
        np.save(os.path.join(SAVE_DIR, f'ig_input_yr{yr}_{out_name}.npy'), yearly_inputs[yr])

    print(f'  Raw arrays saved to {SAVE_DIR}')

    print(f'\n  Generating visualizations...')
    for yr in sorted(yearly_attr.keys()):
        plot_year_panel(yr, yearly_inputs[yr], yearly_attr[yr], meta, out_name)

    plot_grand_average(yearly_attr, yearly_inputs, meta, out_name)
    plot_channel_bar_yearly(yearly_attr, meta, out_name)

if __name__ == '__main__':
    # =========================================================================
    # TARGETED RUNS: Only the 3 configurations needed for the paper!
    # =========================================================================

    TARGET_MODELS = [
        # 1. The "Nowcast" Baseline
        {'model_size': 'normal', 'level': 'real', 'channel_cfg': 'all', 'pred_mode': 'normal', 'criterion': 'MSE'},

        # 2. The Operational Forecast (1-Year-Ahead)
        {'model_size': 'small', 'level': 'real', 'channel_cfg': 'all', 'pred_mode': 'one_year_ahead', 'criterion': 'MSE'},

        # 3. The Biological Lag Proof (5-Year Lag)
        {'model_size': 'normal', 'level': 'real', 'channel_cfg': 'sp_temp', 'pred_mode': 'lag5', 'criterion': 'MSE'},

        #4. Temperature only:
        {'model_size': 'small', 'level': 'real', 'channel_cfg': 'temp_only', 'pred_mode': 'normal', 'criterion': 'MSE'}
    ]

    for run_params in TARGET_MODELS:
        run_targeted_ig(**run_params, phase='TEST', n_steps=50)

Looking at the model performances

In [21]:
"""
analyze_results.py
==================
Produces a paper-ready results table matching the format:

  Model Size | Channel | Prediction Mode | Loss | MAE | Spearman |
  Normalized MAE | Normalized Spearman | Composite

Normalization is done across ALL real-data configurations together
(not separated by pred_mode), with two separate normalization pools:
  - Now-cast + One-year-ahead configs (lag=0)
  - 5-year-lag configs (lag=5)

The climatological baseline rows are inserted as headers for each group
but are NOT included in the normalization pool.

Adjust RESULTS_CSV, OUT_DIR, and BASELINE_* at the top, then run.
"""

import numpy as np
import pandas as pd
from pathlib import Path


# =============================================================================
# CONFIG
# =============================================================================

RESULTS_CSV = '/content/drive/MyDrive/Teleconnection_ViT/analysis/full_results.csv'
OUT_DIR     = '/content/drive/MyDrive/Teleconnection_ViT/analysis'

# Baseline values from baseline_evaluation.py
# Update after re-running baseline_evaluation.py with threshold applied
BASELINE_LAG0 = {'mae': 496.5, 'spearman': 0.8074,
                 'per_year': {2021: 670.5, 2022: 515.2, 2023: 303.7}}
BASELINE_LAG5 = {'mae': 451.3, 'spearman': 0.8128,
                 'per_year': {2016: 597.8, 2017: 479.5, 2018: 276.7}}
TOP_N       = 3          # top configs per pred_mode to include in table
YEAR_OFFSET = 1988

# Display name mappings
MODEL_SIZE_NAMES = {
    'normal': 'Base',
    'small':  'Reduced',
}

CHANNEL_NAMES = {
    'all':           'All',
    'sp_rec':        'Spawner + Recruit',
    'sp_temp':       'Spawner + Temperature',
    'rec_temp':      'Recruit + Temperature',
    'spawners_only': 'Spawner',
    'recruits_only': 'Recruit',
    'temp_only':     'Temperature',
}

PRED_MODE_NAMES = {
    'normal':         'Now-cast',
    'one_year_ahead': 'One-year-ahead',
    'lag5':           '5-year lag',
}


# =============================================================================
# HELPERS
# =============================================================================

def filter_valid_years(df):
    is_real     = df['level'] == 'real'
    is_lag5     = df['pred_mode'] == 'lag5'
    is_test     = df['phase'] == 'TEST'
    is_not_test = ~is_test

    valid_real_lag5   = is_test & is_real &  is_lag5 & df['year'].isin([28, 29, 30])
    valid_real_normal = is_test & is_real & ~is_lag5 & df['year'].isin([33, 34, 35])
    valid_dummy       = is_test & ~is_real            & df['year'].isin([27, 28, 29])

    return df[is_not_test | valid_real_lag5 | valid_real_normal | valid_dummy].copy()


def get_top_configs(summary, pred_modes, top_n):
    """Return top_n configs per pred_mode from the summary dataframe."""
    rows = []
    for pm in pred_modes:
        sub = summary[summary['pred_mode'] == pm]
        rows.append(sub.nlargest(top_n, 'composite'))
    return pd.concat(rows, ignore_index=True)


def normalize_and_composite(sub):
    """Add norm_mae, norm_spearman, composite columns to a subset dataframe."""
    sub = sub.copy()
    mae_min, mae_max = sub['thresh_mae'].min(),      sub['thresh_mae'].max()
    sp_min,  sp_max  = sub['thresh_spearman'].min(), sub['thresh_spearman'].max()

    sub['norm_mae']      = 1 - (sub['thresh_mae'] - mae_min) / (mae_max - mae_min + 1e-10)
    sub['norm_spearman'] = (sub['thresh_spearman'] - sp_min) / (sp_max - sp_min + 1e-10)
    sub['composite']     = (sub['norm_mae'] * sub['norm_spearman']) ** 0.5
    return sub


def make_display_row(row):
    """Convert a summary row to the paper table format."""
    return {
        'Model Size':          MODEL_SIZE_NAMES.get(row['model_size'], row['model_size']),
        'Channel':             CHANNEL_NAMES.get(row['channel_cfg'], row['channel_cfg']),
        'Prediction Mode':     PRED_MODE_NAMES.get(row['pred_mode'], row['pred_mode']),
        'Loss':                row['criterion'],
        'MAE':                 round(row['thresh_mae'], 0),
        'Spearman':            round(row['thresh_spearman'], 3),
        'Normalized MAE':      round(row['norm_mae'], 3),
        'Normalized Spearman': round(row['norm_spearman'], 3),
        'Composite':           round(row['composite'], 3),
    }


def make_baseline_row(label, pred_mode_label, mae, spearman):
    return {
        'Model Size':          'Climatological Baseline',
        'Channel':             'N/A',
        'Prediction Mode':     pred_mode_label,
        'Loss':                'N/A',
        'MAE':                 round(mae, 0),
        'Spearman':            round(spearman, 3),
        'Normalized MAE':      'N/A',
        'Normalized Spearman': 'N/A',
        'Composite':           'N/A',
    }


# =============================================================================
# MAIN
# =============================================================================

out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

# Load and filter
df = pd.read_csv(RESULTS_CSV)
df = filter_valid_years(df)

real_test = df[(df['level'] == 'real') & (df['phase'] == 'TEST')].copy()
print(f"Real test rows : {len(real_test)}")
print(f"Test years     : {sorted((real_test['year'] + YEAR_OFFSET).unique())}")

# Summarize: mean across bootstraps × years per config
group_cols = ['model_size', 'level', 'channel_cfg', 'pred_mode', 'criterion']
summary = (real_test.groupby(group_cols)[['thresh_mae', 'thresh_spearman']]
                    .mean()
                    .reset_index())

# ── Group 1: Now-cast + One-year-ahead (lag=0) ────────────────────────────────
# Normalize ALL lag-0 configs together
lag0_summary = normalize_and_composite(
    summary[summary['pred_mode'].isin(['normal', 'one_year_ahead'])].copy()
)

# Top 3 per pred_mode within this pool
lag0_top = get_top_configs(lag0_summary, ['normal', 'one_year_ahead'], TOP_N)

# ── Group 2: 5-year lag ───────────────────────────────────────────────────────
lag5_summary = normalize_and_composite(
    summary[summary['pred_mode'] == 'lag5'].copy()
)

lag5_top = lag5_summary.nlargest(TOP_N, 'composite')

# ── Print summaries ───────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("NOW-CAST + ONE-YEAR-AHEAD (normalized together)")
print("=" * 80)
print(lag0_top[['model_size', 'channel_cfg', 'pred_mode', 'criterion',
                'thresh_mae', 'thresh_spearman', 'norm_mae', 'norm_spearman', 'composite']]
      .sort_values('composite', ascending=False).round(4).to_string(index=False))

print("\n" + "=" * 80)
print("5-YEAR LAG")
print("=" * 80)
print(lag5_top[['model_size', 'channel_cfg', 'pred_mode', 'criterion',
                'thresh_mae', 'thresh_spearman', 'norm_mae', 'norm_spearman', 'composite']]
      .sort_values('composite', ascending=False).round(4).to_string(index=False))

# ── Build paper table ─────────────────────────────────────────────────────────
table_rows = []

# Baseline row for lag-0 group
table_rows.append(make_baseline_row(
    'lag0', 'Now-cast + One-year-ahead',
    BASELINE_LAG0['mae'], BASELINE_LAG0['spearman']
))

# Now-cast top 3 (sorted by composite desc)
for _, row in (lag0_top[lag0_top['pred_mode'] == 'normal']
               .sort_values('composite', ascending=False).iterrows()):
    table_rows.append(make_display_row(row))

# One-year-ahead top 3
for _, row in (lag0_top[lag0_top['pred_mode'] == 'one_year_ahead']
               .sort_values('composite', ascending=False).iterrows()):
    table_rows.append(make_display_row(row))

# Baseline row for lag-5 group
table_rows.append(make_baseline_row(
    'lag5', '5-year lag',
    BASELINE_LAG5['mae'], BASELINE_LAG5['spearman']
))

# Lag-5 top configs
for _, row in lag5_top.sort_values('composite', ascending=False).iterrows():
    table_rows.append(make_display_row(row))

table_df = pd.DataFrame(table_rows)

# ── Print and save ────────────────────────────────────────────────────────────
print("\n" + "=" * 100)
print("PAPER TABLE")
print("=" * 100)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 50)
print(table_df.to_string(index=False))
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_rows')

out_path = out_dir / 'results_table.csv'
table_df.to_csv(out_path, index=False)
print(f"\nSaved → {out_path}")

# Also save full composite scores for reference
lag0_summary.round(4).to_csv(out_dir / 'composite_scores_lag0.csv', index=False)
lag5_summary.round(4).to_csv(out_dir / 'composite_scores_lag5.csv', index=False)
print(f"Saved → {out_dir / 'composite_scores_lag0.csv'}")
print(f"Saved → {out_dir / 'composite_scores_lag5.csv'}")

# ── Per-year breakdown for top configs ───────────────────────────────────────
df_cal = real_test.copy()
df_cal['calendar_year'] = df_cal['year'] + YEAR_OFFSET

all_top = pd.concat([lag0_top, lag5_top], ignore_index=True)
per_year_rows = []

print("\n" + "=" * 80)
print("PER-YEAR METRICS FOR TOP CONFIGS")
print("=" * 80)

for _, cfg_row in all_top.iterrows():
    mask = (
        (df_cal['model_size']  == cfg_row['model_size'])  &
        (df_cal['channel_cfg'] == cfg_row['channel_cfg']) &
        (df_cal['pred_mode']   == cfg_row['pred_mode'])   &
        (df_cal['criterion']   == cfg_row['criterion'])
    )
    cfg_data = df_cal[mask]
    per_year = (cfg_data.groupby('calendar_year')
                        .agg(mae=('thresh_mae', 'mean'),
                             mae_sd=('thresh_mae', 'std'),
                             spearman=('thresh_spearman', 'mean'),
                             spear_sd=('thresh_spearman', 'std'),
                             n=('thresh_mae', 'count'))
                        .reset_index())

    label = (f"{MODEL_SIZE_NAMES.get(cfg_row['model_size'])} | "
             f"{CHANNEL_NAMES.get(cfg_row['channel_cfg'])} | "
             f"{PRED_MODE_NAMES.get(cfg_row['pred_mode'])} | "
             f"{cfg_row['criterion']}  (composite={cfg_row['composite']:.3f})")
    print(f"\n  {label}")
    print(f"  {'Year':<8} {'MAE':>8} {'±':>6} {'Spearman':>10} {'±':>7} {'n':>5}")
    print(f"  {'-'*48}")
    for _, r in per_year.iterrows():
        print(f"  {int(r['calendar_year']):<8} {r['mae']:>8.1f} "
              f"{r['mae_sd']:>6.1f} {r['spearman']:>10.4f} "
              f"{r['spear_sd']:>7.4f} {int(r['n']):>5}")
        per_year_rows.append({
            'model_size':    cfg_row['model_size'],
            'channel_cfg':   cfg_row['channel_cfg'],
            'pred_mode':     PRED_MODE_NAMES.get(cfg_row['pred_mode']),
            'criterion':     cfg_row['criterion'],
            'composite':     round(cfg_row['composite'], 3),
            'calendar_year': int(r['calendar_year']),
            'mae':           round(r['mae'], 1),
            'mae_sd':        round(r['mae_sd'], 1),
            'spearman':      round(r['spearman'], 4),
            'spear_sd':      round(r['spear_sd'], 4),
            'n_bootstraps':  int(r['n']),
        })

pd.DataFrame(per_year_rows).to_csv(out_dir / 'top_configs_per_year.csv', index=False)
print(f"\nSaved → {out_dir / 'top_configs_per_year.csv'}")

# ── Baseline per-year table ───────────────────────────────────────────────────
# Loads the per-year CSVs saved by baseline_evaluation.py.
# Run baseline_evaluation.py with --lag 0 and --lag 5 first.

print("\n" + "=" * 80)
print("BASELINE PER-YEAR METRICS")
print("=" * 80)

baseline_per_year_rows = []
for lag in [0, 5]:
    # Try loading from OUT_DIR first, then current directory
    for search_dir in [OUT_DIR, '.']:
        csv_path = Path(search_dir) / f'baseline_per_year_lag{lag}.csv'
        if csv_path.exists():
            bdf = pd.read_csv(csv_path)
            print(f"\n  Lag {lag}:")
            print(f"  {'Year':<8} {'MAE':>8} {'±':>6} {'Spearman':>10} {'±':>7} {'n':>5}")
            print(f"  {'-'*48}")
            for _, r in bdf.iterrows():
                print(f"  {int(r['calendar_year']):<8} {r['mae_mean']:>8.1f} "
                      f"{r['mae_sd']:>6.1f} {r['spearman_mean']:>10.4f} "
                      f"{r['spearman_sd']:>7.4f} {int(r['n_bootstraps']):>5}")
                baseline_per_year_rows.append({
                    'source':        f'Baseline (lag={lag})',
                    'lag':           lag,
                    'calendar_year': int(r['calendar_year']),
                    'mae_mean':      r['mae_mean'],
                    'mae_sd':        r['mae_sd'],
                    'spearman_mean': r['spearman_mean'],
                    'spearman_sd':   r['spearman_sd'],
                    'n_bootstraps':  int(r['n_bootstraps']),
                })
            break
    else:
        print(f"\n  Lag {lag}: baseline_per_year_lag{lag}.csv not found — "
              f"run baseline_evaluation.py --lag {lag} first.")

if baseline_per_year_rows:
    baseline_per_year_df = pd.DataFrame(baseline_per_year_rows)
    baseline_per_year_df.to_csv(out_dir / 'baseline_per_year_all.csv', index=False)
    print(f"\nSaved → {out_dir / 'baseline_per_year_all.csv'}")

    # Combined table: model per-year + baseline per-year for easy comparison
    model_per_year_df = pd.DataFrame(per_year_rows)
    model_per_year_df['source'] = (
        model_per_year_df['model_size'] + ' | ' +
        model_per_year_df['channel_cfg'] + ' | ' +
        model_per_year_df['pred_mode']   + ' | ' +
        model_per_year_df['criterion']
    )
    combined_cols = ['source', 'calendar_year', 'mae_mean', 'mae_sd',
                     'spearman_mean', 'spearman_sd', 'n_bootstraps']
    # Rename model cols to match baseline cols
    model_renamed = model_per_year_df.rename(columns={
        'mae': 'mae_mean', 'mae_sd': 'mae_sd',
        'spearman': 'spearman_mean', 'spear_sd': 'spearman_sd',
        'n_bootstraps': 'n_bootstraps',
    })[combined_cols]
    combined_df = pd.concat(
        [baseline_per_year_df[combined_cols], model_renamed],
        ignore_index=True
    )
    combined_df.to_csv(out_dir / 'per_year_combined.csv', index=False)
    print(f"Saved → {out_dir / 'per_year_combined.csv'}")
print(f"\nDone.")

Real test rows : 25200
Test years     : [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2021), np.int64(2022), np.int64(2023)]

NOW-CAST + ONE-YEAR-AHEAD (normalized together)
model_size channel_cfg      pred_mode criterion  thresh_mae  thresh_spearman  norm_mae  norm_spearman  composite
    normal         all         normal       MSE    331.5294           0.8664    0.9644         0.9602     0.9623
    normal     sp_temp         normal       MSE    308.8401           0.8601    0.9817         0.9377     0.9594
     small   temp_only         normal       MSE    375.6244           0.8686    0.9308         0.9680     0.9492
     small         all one_year_ahead       MSE    340.5604           0.8483    0.9575         0.8956     0.9260
    normal     sp_temp one_year_ahead       MSE    381.3760           0.8547    0.9264         0.9183     0.9224
     small    rec_temp one_year_ahead       MSE    459.3004           0.8685    0.8671         0.9676     0.9160

5-YEAR LAG
model_size 

In [15]:
"""
training_dynamics_table.py
===========================
Generates a summary table of training dynamics for all trained models,
supporting claims in Section 2.3.3 about overfitting ratios, best validation
epochs, and differences between real and synthetic data.

For each model, extracts from training_history.json:
  - Final training loss (epoch 20)
  - Minimum validation loss
  - Best validation epoch
  - Final validation loss (epoch 20)
  - Overfit ratio: final_val_loss / final_train_loss
  - Val/train ratio at best epoch

Outputs
-------
  training_dynamics_full.csv    — one row per model, all configs
  training_dynamics_real.csv    — real data only
  training_dynamics_dummy.csv   — synthetic data only
  training_dynamics_summary.txt — summary statistics supporting Section 2.3.3

Usage
-----
    python training_dynamics_table.py

    # Adjust DRIVE_BASE to your model_outputs directory
"""

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path


# =============================================================================
# CONFIG
# =============================================================================

DRIVE_BASE = '/content/drive/MyDrive/Teleconnection_ViT/model_outputs'
OUT_DIR    = '/content/drive/MyDrive/Teleconnection_ViT/analysis'


# =============================================================================
# MAIN
# =============================================================================

def extract_training_dynamics(run_dir: str) -> dict | None:
    """
    Extract training dynamics from one training_history.json.
    Returns None if the file is missing or malformed.
    """
    history_path = os.path.join(run_dir, 'training_history.json')
    if not os.path.exists(history_path):
        return None

    with open(history_path) as f:
        hist = json.load(f)

    train_loss = hist.get('train_loss', [])
    val_loss   = hist.get('val_loss',   [])

    if not train_loss or not val_loss:
        return None

    # Parse path
    rel   = os.path.relpath(run_dir, DRIVE_BASE)
    parts = rel.split(os.sep)
    if len(parts) != 5:
        return None
    model_size, level, channel_cfg, pred_mode, criterion = parts
    data_type = 'real' if level == 'real' else 'dummy'

    best_epoch     = int(np.argmin(val_loss)) + 1   # 1-indexed
    best_val_loss  = float(min(val_loss))
    final_train    = float(train_loss[-1])
    final_val      = float(val_loss[-1])
    initial_val    = float(val_loss[0])
    overfit_ratio  = final_val / (final_train + 1e-10)
    val_improvement = initial_val - best_val_loss   # how much val improved

    # Val loss at best epoch vs train loss at best epoch
    train_at_best  = float(train_loss[best_epoch - 1])
    val_at_best    = best_val_loss
    ratio_at_best  = val_at_best / (train_at_best + 1e-10)

    n_epochs = len(train_loss)

    return {
        'model_size':       model_size,
        'level':            level,
        'data_type':        data_type,
        'channel_cfg':      channel_cfg,
        'pred_mode':        pred_mode,
        'criterion':        criterion,
        'n_epochs':         n_epochs,
        'final_train_loss': round(final_train,   4),
        'final_val_loss':   round(final_val,     4),
        'best_val_loss':    round(best_val_loss, 4),
        'best_epoch':       best_epoch,
        'initial_val_loss': round(initial_val,   4),
        'val_improvement':  round(val_improvement, 4),
        'overfit_ratio':    round(overfit_ratio, 2),   # final_val / final_train
        'ratio_at_best':    round(ratio_at_best, 2),   # best_val / train_at_best
        'train_at_best':    round(train_at_best, 4),
    }


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    rows = []

    for root, dirs, files in os.walk(DRIVE_BASE):
        if 'training_history.json' not in files:
            continue
        rel   = os.path.relpath(root, DRIVE_BASE)
        parts = rel.split(os.sep)
        if len(parts) != 5:
            continue
        result = extract_training_dynamics(root)
        if result:
            rows.append(result)

    if not rows:
        print("No training history files found.")
        return

    df = pd.DataFrame(rows)
    df = df.sort_values(['data_type', 'level', 'model_size',
                         'channel_cfg', 'pred_mode', 'criterion'])

    # Save full table
    full_path = os.path.join(OUT_DIR, 'training_dynamics_full.csv')
    df.to_csv(full_path, index=False)
    print(f"Saved full table → {full_path}  ({len(df)} rows)")

    # Real data only
    real_df = df[df['data_type'] == 'real'].copy()
    real_path = os.path.join(OUT_DIR, 'training_dynamics_real.csv')
    real_df.to_csv(real_path, index=False)
    print(f"Saved real table → {real_path}  ({len(real_df)} rows)")

    # Dummy data only
    dummy_df = df[df['data_type'] == 'dummy'].copy()
    dummy_path = os.path.join(OUT_DIR, 'training_dynamics_dummy.csv')
    dummy_df.to_csv(dummy_path, index=False)
    print(f"Saved dummy table → {dummy_path}  ({len(dummy_df)} rows)")

    # ── Summary statistics ────────────────────────────────────────────────────
    summary_lines = []
    summary_lines.append("=" * 72)
    summary_lines.append("TRAINING DYNAMICS SUMMARY")
    summary_lines.append("=" * 72)

    for data_type in ['real', 'dummy']:
        sub = df[df['data_type'] == data_type]
        if sub.empty:
            continue

        summary_lines.append(f"\n{'─'*72}")
        summary_lines.append(f"DATA TYPE: {data_type.upper()}")
        summary_lines.append(f"{'─'*72}")

        for criterion in ['MSE', 'Tweedie']:
            csub = sub[sub['criterion'] == criterion]
            if csub.empty:
                continue

            summary_lines.append(f"\n  Criterion: {criterion}  (n={len(csub)} runs)")
            summary_lines.append(
                f"  Final train loss    : "
                f"{csub['final_train_loss'].min():.4f} – {csub['final_train_loss'].max():.4f}  "
                f"(mean={csub['final_train_loss'].mean():.4f})"
            )
            summary_lines.append(
                f"  Final val loss      : "
                f"{csub['final_val_loss'].min():.4f} – {csub['final_val_loss'].max():.4f}  "
                f"(mean={csub['final_val_loss'].mean():.4f})"
            )
            summary_lines.append(
                f"  Best val loss       : "
                f"{csub['best_val_loss'].min():.4f} – {csub['best_val_loss'].max():.4f}  "
                f"(mean={csub['best_val_loss'].mean():.4f})"
            )
            summary_lines.append(
                f"  Best epoch          : "
                f"{int(csub['best_epoch'].min())} – {int(csub['best_epoch'].max())}  "
                f"(mean={csub['best_epoch'].mean():.1f})"
            )
            summary_lines.append(
                f"  Overfit ratio       : "
                f"{csub['overfit_ratio'].min():.2f}x – {csub['overfit_ratio'].max():.2f}x  "
                f"(mean={csub['overfit_ratio'].mean():.2f}x)"
            )

        # Break down by model size for real data
        if data_type == 'real':
            summary_lines.append(f"\n  Overfit ratio by model size (MSE only):")
            mse_sub = sub[sub['criterion'] == 'MSE']
            for ms in ['normal', 'small']:
                ms_sub = mse_sub[mse_sub['model_size'] == ms]
                if ms_sub.empty:
                    continue
                summary_lines.append(
                    f"    {ms:8s}: "
                    f"{ms_sub['overfit_ratio'].min():.1f}x – {ms_sub['overfit_ratio'].max():.1f}x  "
                    f"(mean={ms_sub['overfit_ratio'].mean():.1f}x)"
                )

            summary_lines.append(f"\n  Best epoch by model size (MSE only):")
            for ms in ['normal', 'small']:
                ms_sub = mse_sub[mse_sub['model_size'] == ms]
                if ms_sub.empty:
                    continue
                summary_lines.append(
                    f"    {ms:8s}: "
                    f"epoch {int(ms_sub['best_epoch'].min())} – {int(ms_sub['best_epoch'].max())}  "
                    f"(mean={ms_sub['best_epoch'].mean():.1f})"
                )

            # Highlight the most extreme overfit
            most_overfit = sub[sub['criterion'] == 'MSE'].nlargest(5, 'overfit_ratio')
            summary_lines.append(f"\n  Top 5 most overfit configs (MSE):")
            for _, row in most_overfit.iterrows():
                summary_lines.append(
                    f"    {row['model_size']:6s} {row['channel_cfg']:15s} "
                    f"{row['pred_mode']:15s}  "
                    f"overfit={row['overfit_ratio']:.1f}x  "
                    f"best_epoch={int(row['best_epoch'])}"
                )

        if data_type == 'dummy':
            summary_lines.append(f"\n  Val/train ratio at best epoch by model size:")
            for ms in ['normal', 'small']:
                ms_sub = sub[sub['model_size'] == ms]
                if ms_sub.empty:
                    continue
                summary_lines.append(
                    f"    {ms:8s}: "
                    f"{ms_sub['ratio_at_best'].min():.3f}x – "
                    f"{ms_sub['ratio_at_best'].max():.3f}x  "
                    f"(mean={ms_sub['ratio_at_best'].mean():.3f}x)"
                )

    summary_lines.append("\n" + "=" * 72)

    summary_text = "\n".join(summary_lines)
    print(summary_text)

    summary_path = os.path.join(OUT_DIR, 'training_dynamics_summary.txt')
    with open(summary_path, 'w') as f:
        f.write(summary_text)
    print(f"\nSaved summary → {summary_path}")

    # ── Full printout of real data table ─────────────────────────────────────
    print("\n" + "=" * 100)
    print("REAL DATA — FULL TRAINING DYNAMICS TABLE")
    print("=" * 100)
    pd.set_option('display.max_rows', 200)
    pd.set_option('display.width', 200)
    print(real_df[['model_size', 'channel_cfg', 'pred_mode', 'criterion',
                   'final_train_loss', 'final_val_loss', 'best_val_loss',
                   'best_epoch', 'overfit_ratio']]
          .to_string(index=False))
    pd.reset_option('display.max_rows')
    pd.reset_option('display.width')


if __name__ == '__main__':
    main()

Saved full table → /content/drive/MyDrive/Teleconnection_ViT/analysis/training_dynamics_full.csv  (144 rows)
Saved real table → /content/drive/MyDrive/Teleconnection_ViT/analysis/training_dynamics_real.csv  (84 rows)
Saved dummy table → /content/drive/MyDrive/Teleconnection_ViT/analysis/training_dynamics_dummy.csv  (60 rows)
TRAINING DYNAMICS SUMMARY

────────────────────────────────────────────────────────────────────────
DATA TYPE: REAL
────────────────────────────────────────────────────────────────────────

  Criterion: MSE  (n=40 runs)
  Final train loss    : 0.1131 – 0.1690  (mean=0.1381)
  Final val loss      : 2.0632 – 3.1141  (mean=2.6318)
  Best val loss       : 1.7256 – 2.8851  (mean=2.3113)
  Best epoch          : 1 – 16  (mean=6.7)
  Overfit ratio       : 12.77x – 26.57x  (mean=19.32x)

  Criterion: Tweedie  (n=40 runs)
  Final train loss    : 1427.1618 – 1552.5204  (mean=1490.0532)
  Final val loss      : 1883.2527 – 2158.3425  (mean=2006.3814)
  Best val loss       : 185

In [36]:
"""
top_model_figure.py
====================
Creates a 7-column spatial + line-plot summary figure for top CrabTransformer runs.

Row order
---------
  0  Climatological baseline  (lag-0)
  1  All channels | Now-cast | MSE | Base-size
  2  Temperature only | Now-cast | MSE | Reduced-size
  3  Recruits + Temp | One-year-ahead | MSE | Reduced-size
  --- section divider + new column headers ---
  4  Climatological baseline  (lag-5)
  5  Spawners + Temp | Lag-5 | MSE | Base-size

Column headers differ between sections:
  Lag-0 section : Spawner history (t-1,...,t-5) / Spawner current (t) / ...
  Lag-5 section : Spawner history (t-6,...,t-10) / Spawner current (t-5) / ...

Changes from previous version
------------------------------
- Row order updated as above
- temp_only now-cast uses MSE (not Tweedie)
- Section divider + second set of column headers for lag-5 rows
- Line plot legend: removed dashed-line entry
- Line plot title updated to include "Median across bootstrap"
"""

import os
import sys
import json
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_DIR   = '/content/Teleconnections-ViT'
DRIVE_BASE = '/content/drive/MyDrive/Teleconnection_ViT/model_outputs'
SAVE_DIR   = '/content/drive/MyDrive/Teleconnection_ViT/analysis'

MEMORY_YEARS    = 5
BATCH_SIZE      = 8
DATA_START_YEAR = 1988

sys.path.insert(0, REPO_DIR)
from models.model     import CrabTransformer
from data.data_helper import get_dataloaders

os.makedirs(SAVE_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Row colours ───────────────────────────────────────────────────────────────
ROW_COLORS = [
    '#e41a1c', '#377eb8', '#4daf4a', '#984ea3',
    '#ff7f00', '#a65628', '#f781bf', '#555555',
]

# ── Runs to display ───────────────────────────────────────────────────────────
TARGET_RUNS = [
    ('BASELINE', 'real', 'climatological', 'normal',         'N/A'),  # lag-0 baseline
    ('normal',   'real', 'all',            'normal',         'MSE'),  # all channels now-cast
    ('small',    'real', 'temp_only',      'normal',         'MSE'),  # temp only now-cast MSE
    ('small',    'real', 'rec_temp',       'one_year_ahead', 'MSE'),  # rec+temp 1yr-ahead
    ('BASELINE', 'real', 'climatological', 'lag5',           'N/A'),  # lag-5 baseline
    ('normal',   'real', 'sp_temp',        'lag5',           'MSE'),  # sp+temp lag-5
]

# First row index of the lag-5 section (section divider drawn above this row)
LAG5_SECTION_START = 4

RUN_DISPLAY_NAMES = {
    ('BASELINE', 'real', 'climatological', 'normal',         'N/A'): 'Climatological baseline | Now-cast + 1-yr ahead',
    ('normal',   'real', 'all',            'normal',         'MSE'): 'All channels | Now-cast | MSE | Base-size',
    ('small',    'real', 'temp_only',      'normal',         'MSE'): 'Bottom Temp only | Now-cast | MSE | Reduced-size',
    ('small',    'real', 'rec_temp',       'one_year_ahead', 'MSE'): 'Recruits + Bottom Temp | 1-yr ahead | MSE | Reduced-size',
    ('BASELINE', 'real', 'climatological', 'lag5',           'N/A'): 'Climatological baseline | Lag-5',
    ('normal',   'real', 'sp_temp',        'lag5',           'MSE'): 'Spawners + Bottom Temp | Lag-5 | MSE | Base-size',
}

# Column titles for lag-0 section
COL_TITLES_LAG0 = [
    'Spawner history\n(t-1,...,t-5)',
    'Spawner\ncurrent (t)',
    'Bottom Temperature\nhistory (t-1,...,t-5)',
    'Bottom Temperature\ncurrent (t)',
    'Recruit history\n(t-1,...,t-5)',
    'Observed\nrecruit',
    'Predicted\nrecruit',
]

# Column titles for lag-5 section
COL_TITLES_LAG5 = [
    'Spawner history\n(t-6,...,t-10)',
    'Spawner\ncurrent (t-5)',
    'Bottom Temperature\nhistory (t-6,...,t-10)',
    'Bottom Temperature\ncurrent (t-5)',
    'Recruit history\n(t-1,...,t-5)',
    'Observed\nrecruit',
    'Predicted\nrecruit',
]

CMAPS = {'spawner': 'YlOrRd', 'temp': 'RdBu_r', 'recruit': 'Blues'}


# ============================================================
#  PIPELINE HELPERS
# ============================================================

def get_year_splits(data_type, lag):
    if data_type == 'real':
        return (24, 8, 4) if lag == 0 else (21, 6, 4)
    return (18, 9, 3)


def load_run(model_size, level, channel_cfg, pred_mode, criterion):
    data_type = 'real' if level == 'real' else 'dummy'
    run_dir   = os.path.join(DRIVE_BASE, model_size, level,
                             channel_cfg, pred_mode, criterion)

    with open(os.path.join(run_dir, 'training_history.json')) as f:
        hist = json.load(f)

    meta = hist['channel_cfg_meta']
    bias = hist.get('bias_correction', 1.0)
    t_yr, v_yr, te_yr = get_year_splits(data_type, meta['lag'])

    tr_ld, va_ld, te_ld = get_dataloaders(
        batch_size=BATCH_SIZE, memory_years=MEMORY_YEARS,
        train_years=t_yr, val_years=v_yr, test_years=te_yr,
        level=level, data_type=data_type,
        include_current_spawner=meta['incl_curr'],
        lag=meta['lag'],
        use_temp=meta['use_temp'],
        use_spawners=meta['use_spawners'],
        use_recruits=meta['use_recruits'],
    )

    model = CrabTransformer(
        grid_size=50, patch_size=5,
        in_channels=meta['in_channels'],
        embed_dim=meta['embed_dim'],
        num_heads=meta['num_heads'],
        num_layers=meta['num_layers'],
        d_ff=meta['d_ff'],
        dropout=0.0,
        channel_mask_indices=meta['channel_mask_indices'],
    ).to(DEVICE)
    model.load_state_dict(
        torch.load(os.path.join(run_dir, 'best_model.pt'), map_location=DEVICE)
    )
    model.eval()

    return model, meta, bias, t_yr, v_yr, te_yr, tr_ld, va_ld, te_ld


def extract_channels(inp, meta):
    ch   = {k: None for k in ('sp_curr', 'sp_hist',
                               'temp_curr', 'temp_hist', 'rec_hist')}
    idx  = 0
    incl = meta['incl_curr']

    if meta['use_spawners']:
        if incl:
            ch['sp_curr'] = inp[idx];  idx += 1
        ch['sp_hist'] = [inp[idx + k] for k in range(5)];  idx += 5

    if meta['use_recruits']:
        ch['rec_hist'] = [inp[idx + k] for k in range(5)];  idx += 5

    if meta['use_temp']:
        if incl:
            ch['temp_curr'] = inp[idx];  idx += 1
        ch['temp_hist'] = [inp[idx + k] for k in range(5)];  idx += 5

    return ch


def get_test_sample(test_loader, model, bias, valid_mask, target_year_idx=None):
    best_yr   = -1
    best_data = (None, None, None)

    with torch.no_grad():
        for batch in test_loader:
            inputs, targets, temporal_mask, year_idx, spatial_mask, valid_year = batch
            for i in range(inputs.shape[0]):
                if valid_year[i] == 0:
                    continue
                yi = int(year_idx[i])
                if target_year_idx is not None and yi != target_year_idx:
                    continue

                pred = model(
                    inputs[i:i+1].to(DEVICE),
                    year_idx[i:i+1].to(DEVICE),
                    temporal_mask[i:i+1].to(DEVICE),
                    spatial_mask=spatial_mask[i:i+1].to(DEVICE),
                )
                pred_log     = pred[0, 0].cpu().numpy()
                pred_display = np.log1p(
                    np.clip(np.exp(pred_log) * bias - 1.0, 0.0, None)
                )
                pred_display[~valid_mask] = 0.0
                target_log = targets[i, 0].numpy().copy()
                target_log[~valid_mask] = 0.0
                print(f'    Sample year_idx={yi}  '
                      f'inp mean={inputs[i].mean():.3f}  '
                      f'tgt mean={target_log[valid_mask].mean():.3f}  '
                      f'pred mean={pred_display[valid_mask].mean():.3f}')

                if target_year_idx is not None:
                    return inputs[i].numpy(), target_log, pred_display

                if yi > best_yr:
                    best_yr   = yi
                    best_data = (inputs[i].numpy(), target_log, pred_display)

    return best_data


def collect_yearly_aggregates(loaders, model, bias, valid_mask):
    from collections import defaultdict
    agg = defaultdict(lambda: {'pred': [], 'obs': []})
    with torch.no_grad():
        for loader in loaders:
            for batch in loader:
                inputs, targets, temporal_mask, year_idx, spatial_mask, valid_year = batch
                preds = model(
                    inputs.to(DEVICE),
                    year_idx.to(DEVICE),
                    temporal_mask.to(DEVICE),
                    spatial_mask=spatial_mask.to(DEVICE),
                )
                for i in range(preds.shape[0]):
                    if valid_year[i] == 0:
                        continue
                    yr    = int(year_idx[i])
                    p_raw = np.clip(
                        np.exp(preds[i, 0].cpu().numpy()) * bias - 1.0, 0.0, None
                    )
                    t_raw = np.expm1(targets[i, 0].numpy())
                    p_raw[~valid_mask] = 0.0
                    t_raw[~valid_mask] = 0.0
                    agg[yr]['pred'].append(float(p_raw[valid_mask].sum()))
                    agg[yr]['obs'].append(float(t_raw[valid_mask].sum()))
    return agg


# ============================================================
#  DRAWING HELPERS
# ============================================================

def _cell_pos(fig, gs, row, col):
    ax  = fig.add_subplot(gs[row, col])
    pos = ax.get_position()
    fig.delaxes(ax)
    return pos


def _blank_ax(fig, bbox, msg='N/A'):
    ax = fig.add_axes([bbox.x0, bbox.y0, bbox.width, bbox.height])
    ax.set_facecolor('#e8e8e8')
    ax.text(0.5, 0.5, msg, ha='center', va='center',
            transform=ax.transAxes, fontsize=11, color='#000000', style='italic')
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    return ax


def _make_cmap(name):
    import copy
    cmap = copy.copy(plt.get_cmap(name))
    cmap.set_bad('black')
    return cmap


def _masked(image, valid_mask):
    if valid_mask is None:
        return image
    return np.ma.array(image, mask=~valid_mask)


def draw_single_ax(fig, bbox, image, cmap, vmin, vmax, border_color,
                   valid_mask=None):
    if image is None:
        _blank_ax(fig, bbox)
        return
    ax = fig.add_axes([bbox.x0, bbox.y0, bbox.width, bbox.height])
    ax.set_facecolor('black')
    ax.imshow(_masked(image, valid_mask), cmap=_make_cmap(cmap),
              vmin=vmin, vmax=vmax, interpolation='nearest', aspect='auto')
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_edgecolor(border_color)
        sp.set_linewidth(2.5)
        sp.set_visible(True)


def draw_stack_ax(fig, bbox, images, cmap, vmin, vmax, border_color,
                  valid_mask=None):
    valid = [(k, img) for k, img in enumerate(images) if img is not None]
    if not valid:
        _blank_ax(fig, bbox)
        return

    n      = len(valid)
    step_x = bbox.width  * 0.06
    step_y = bbox.height * 0.06
    card_w = bbox.width  - (n - 1) * step_x
    card_h = bbox.height - (n - 1) * step_y
    cmap_obj = _make_cmap(cmap)

    for rank in range(n - 1, -1, -1):
        _, img  = valid[rank]
        offset  = (n - 1 - rank)
        x0 = bbox.x0 + offset * step_x
        y0 = bbox.y0 + offset * step_y

        ax = fig.add_axes([x0, y0, card_w, card_h])
        ax.set_facecolor('black')
        ax.imshow(_masked(img, valid_mask), cmap=cmap_obj,
                  vmin=vmin, vmax=vmax, interpolation='nearest', aspect='auto')
        ax.set_xticks([]); ax.set_yticks([])

        is_front = (rank == 0)
        for sp in ax.spines.values():
            sp.set_edgecolor(border_color)
            sp.set_linewidth(2.5 if is_front else 0.8)
            sp.set_visible(True)


def _load_baseline_grid(lag: int):
    for search_dir in [SAVE_DIR, '.']:
        path = os.path.join(search_dir, f'baseline_mean_grid_lag{lag}.npy')
        if os.path.exists(path):
            return np.load(path).astype(np.float32)
    print(f'  Warning: baseline_mean_grid_lag{lag}.npy not found.')
    return None


def _load_baseline_obs_mean(lag: int):
    for search_dir in [SAVE_DIR, '.']:
        path = os.path.join(search_dir, f'baseline_obs_mean_lag{lag}.npy')
        if os.path.exists(path):
            return np.load(path).astype(np.float32)
    return None


def _get_baseline_obs_single(lag, target_year_idx, valid_mask):
    """Load a single-year single-bootstrap observed target for a baseline row."""
    obs_single = None
    try:
        t_yr_b, v_yr_b, te_yr_b = get_year_splits('real', lag)
        _, _, te_ld_base = get_dataloaders(
            batch_size=BATCH_SIZE, memory_years=MEMORY_YEARS,
            train_years=t_yr_b, val_years=v_yr_b, test_years=te_yr_b,
            level='real', data_type='real',
            include_current_spawner=True,
            lag=lag,
            use_temp=True,
            use_spawners=True,
            use_recruits=True,
        )
        req_yr  = (target_year_idx - lag if target_year_idx is not None else None)
        best_yr = -1
        with torch.no_grad():
            for batch in te_ld_base:
                _, targets, _, year_idx, spatial_mask_b, valid_year = batch
                for i in range(targets.shape[0]):
                    if valid_year[i] == 0:
                        continue
                    yi = int(year_idx[i])
                    if req_yr is not None and yi != req_yr:
                        continue
                    if req_yr is None and yi <= best_yr:
                        continue
                    tgt = targets[i, 0].numpy().copy()
                    tgt[~valid_mask] = 0.0
                    obs_single = tgt
                    best_yr    = yi
                    if req_yr is not None:
                        break
                if req_yr is not None and obs_single is not None:
                    break
    except Exception as e:
        print(f'  Warning: Could not load baseline obs target: {e}')
    return obs_single


# ============================================================
#  MAIN FIGURE
# ============================================================

def make_figure(runs=None, save_path=None, display_year=None,
                title='CrabTransformer — Top Model Summary'):
    if runs is None:
        runs = TARGET_RUNS

    target_year_idx = None
    if display_year is not None:
        target_year_idx = display_year - DATA_START_YEAR
        title = f'{title} ({display_year})'

    n_rows = len(runs)
    n_cols = 7

    # ── Font sizes ────────────────────────────────────────────────────────────
    FS_SUPTITLE    = 18
    FS_SECTION_HDR = 16
    FS_COL_HDR     = 14
    FS_ROW_LABEL   = 14
    FS_PHASE_LABEL = 14
    FS_AXIS_LABEL  = 14
    FS_AXIS_TITLE  = 14
    FS_LEGEND      = 14
    FS_DIVIDER     = 11   # section divider label font size

    # ── Figure size ───────────────────────────────────────────────────────────
    cell_w   = 2.4
    cell_h   = 2.6
    label_w  = 2.0
    line_h   = 5.2
    top_pad  = 0.55
    spacer_w = 0.30
    fig_w    = label_w + cell_w * n_cols + cell_w * spacer_w * 2
    fig_h    = top_pad + cell_h * n_rows + cell_h * 0.45 + line_h + 0.4

    fig = plt.figure(figsize=(fig_w, fig_h), dpi=130)

    left_margin = label_w / fig_w

    outer = gridspec.GridSpec(
        2, 1, figure=fig,
        height_ratios=[top_pad + cell_h * n_rows, line_h],
        hspace=0.08,
        top=0.87, bottom=0.10,
        left=left_margin, right=0.99,
    )

    def _gcol(c):
        if c < 5:  return c
        if c == 5: return c + 1
        return c + 2

    # The gridspec has n_rows + 1 rows: a narrow gap row is inserted at
    # LAG5_SECTION_START to provide physical space for the second column headers.
    # GAP_H controls how tall the gap row is relative to a data row.
    GAP_H       = 0.45
    n_gs_rows   = n_rows + 1   # +1 for the gap row
    gap_gs_row  = LAG5_SECTION_START   # gridspec row index of the gap

    # Build height_ratios: 1.0 for data rows, GAP_H for the gap row
    height_ratios = []
    for gs_r in range(n_gs_rows):
        height_ratios.append(GAP_H if gs_r == gap_gs_row else 1.0)

    grid_top = gridspec.GridSpecFromSubplotSpec(
        n_gs_rows, n_cols + 2,
        subplot_spec=outer[0],
        hspace=0.10, wspace=0.06,
        width_ratios=[1, 1, 1, 1, 1, spacer_w, 1, spacer_w, 1],
        height_ratios=height_ratios,
    )
    ax_line = fig.add_subplot(outer[1])

    # Map data row index → gridspec row index (rows at or after the gap shift by 1)
    def _grows(data_row):
        return data_row if data_row < LAG5_SECTION_START else data_row + 1

    # ── Spatial mask ──────────────────────────────────────────────────────────
    mask_path  = os.path.join(REPO_DIR, 'data/real/output/spatial_mask.npy')
    valid_mask = (np.load(mask_path) > 0) if os.path.exists(mask_path) \
                 else np.ones((50, 50), dtype=bool)

    # ── Pre-compute cell positions using mapped gridspec rows ─────────────────
    cell_bboxes = {}
    for r in range(n_rows):
        for c in range(n_cols):
            cell_bboxes[(r, c)] = _cell_pos(fig, grid_top, _grows(r), _gcol(c))

    # Also get the bbox of the gap row itself (for placing lag-5 column headers)
    gap_bboxes = {}
    for c in range(n_cols):
        gap_bboxes[c] = _cell_pos(fig, grid_top, gap_gs_row, _gcol(c))

    # ── Header geometry ───────────────────────────────────────────────────────
    top_row_top = cell_bboxes[(0, 0)].y1
    fig_hdr_top = 0.96
    hdr_space   = fig_hdr_top - top_row_top

    col_hdr_y = top_row_top + hdr_space * 0.28
    sec_hdr_y = top_row_top + hdr_space * 0.78
    rule_y    = top_row_top + hdr_space * 0.53

    inputs_x0 = cell_bboxes[(0, 0)].x0
    inputs_x1 = cell_bboxes[(0, 4)].x1
    output_x0 = cell_bboxes[(0, 5)].x0
    output_x1 = cell_bboxes[(0, 5)].x1
    target_x0 = cell_bboxes[(0, 6)].x0
    target_x1 = cell_bboxes[(0, 6)].x1
    inputs_cx  = (inputs_x0 + inputs_x1) / 2
    output_cx  = (output_x0 + output_x1) / 2
    target_cx  = (target_x0 + target_x1) / 2
    div_x      = (cell_bboxes[(0, 4)].x1 + cell_bboxes[(0, 5)].x0) / 2
    div2_x     = (cell_bboxes[(0, 5)].x1 + cell_bboxes[(0, 6)].x0) / 2
    grid_bot   = cell_bboxes[(n_rows - 1, 0)].y0

    # ── Column headers — lag-0 section (top) ─────────────────────────────────
    for col, ttl in enumerate(COL_TITLES_LAG0):
        bbox = cell_bboxes[(0, col)]
        fig.text(
            bbox.x0 + bbox.width / 2, col_hdr_y, ttl,
            ha='center', va='center',
            fontsize=FS_COL_HDR, fontweight='bold',
            transform=fig.transFigure,
        )

    # ── Column headers — lag-5 section ───────────────────────────────────────
    # Centred vertically inside the gap row
    for col, ttl in enumerate(COL_TITLES_LAG5):
        bbox = gap_bboxes[col]
        gap_cy = bbox.y0 + bbox.height / 2
        fig.text(
            bbox.x0 + bbox.width / 2, gap_cy, ttl,
            ha='center', va='center',
            fontsize=FS_COL_HDR, fontweight='bold',
            transform=fig.transFigure,
        )

    # ── Per-run processing ────────────────────────────────────────────────────
    line_data = {}
    row_meta  = []

    for row_idx, run_spec in enumerate(runs):
        model_size, level, channel_cfg, pred_mode, criterion = run_spec
        color = ROW_COLORS[row_idx % len(ROW_COLORS)]
        label = RUN_DISPLAY_NAMES.get(
            run_spec,
            f"{channel_cfg} | {pred_mode} | {criterion} ({model_size})",
        )
        print(f'\n[{row_idx+1}/{n_rows}]  {label}')

        # Row label on left margin — wrap on ' | ' so each component is its own line
        bbox0  = cell_bboxes[(row_idx, 0)]
        row_cy = bbox0.y0 + bbox0.height / 2
        label_wrapped = label.replace(' | ', '\n')
        fig.text(
            left_margin - 0.01, row_cy, label_wrapped,
            ha='right', va='center', fontsize=FS_ROW_LABEL,
            transform=fig.transFigure, color=color, fontweight='bold',
            linespacing=1.3,
        )

        # ── BASELINE ROW ──────────────────────────────────────────────────────
        if model_size == 'BASELINE':
            lag = 5 if pred_mode == 'lag5' else 0

            for c in range(5):
                _blank_ax(fig, cell_bboxes[(row_idx, c)], '')

            baseline_grid = _load_baseline_grid(lag)
            obs_mean_grid = _load_baseline_obs_mean(lag)

            if baseline_grid is not None:
                # Col 5 — single-year observed target matching display_year
                obs_single = _get_baseline_obs_single(lag, target_year_idx, valid_mask)

                if obs_single is not None:
                    rec_vmin = float(obs_single[valid_mask].min())
                    rec_vmax = float(obs_single[valid_mask].max())
                    draw_single_ax(fig, cell_bboxes[(row_idx, 5)],
                                   obs_single, CMAPS['recruit'],
                                   rec_vmin, rec_vmax, color, valid_mask)
                elif obs_mean_grid is not None:
                    rec_vmin = float(np.log1p(np.maximum(0, obs_mean_grid[valid_mask])).min())
                    rec_vmax = float(np.log1p(obs_mean_grid[valid_mask]).max())
                    draw_single_ax(fig, cell_bboxes[(row_idx, 5)],
                                   np.log1p(np.maximum(0, obs_mean_grid)),
                                   CMAPS['recruit'], rec_vmin, rec_vmax,
                                   color, valid_mask)
                else:
                    _blank_ax(fig, cell_bboxes[(row_idx, 5)], 'obs target')

                # Col 6 — baseline mean grid (log1p for display)
                pred_log = np.log1p(np.maximum(0, baseline_grid))
                rec_vmin = float(pred_log[valid_mask].min())
                rec_vmax = float(pred_log[valid_mask].max())
                draw_single_ax(fig, cell_bboxes[(row_idx, 6)],
                               pred_log, CMAPS['recruit'],
                               rec_vmin, rec_vmax, color, valid_mask)

                # Line data — flat predicted total across full year range
                pred_total = float(baseline_grid[valid_mask].sum())
                all_year_idxs = []
                for other_idx, other_agg in line_data.items():
                    if runs[other_idx][0] != 'BASELINE':
                        other_lag = row_meta[other_idx][5] if other_idx < len(row_meta) else 0
                        if other_lag == lag:
                            all_year_idxs = sorted(other_agg.keys())
                            break
                if not all_year_idxs:
                    n_years_total = (21 + 6 + 4) if lag == 5 else (24 + 8 + 4)
                    all_year_idxs = list(range(n_years_total))

                line_data[row_idx] = {
                    yi: {'pred': [pred_total], 'obs': [0.0]}
                    for yi in all_year_idxs
                }
            else:
                _blank_ax(fig, cell_bboxes[(row_idx, 5)], 'run baseline_evaluation.py')
                _blank_ax(fig, cell_bboxes[(row_idx, 6)], 'run baseline_evaluation.py')

            t_yr  = 21 if lag == 5 else 24
            v_yr  = 6  if lag == 5 else 8
            te_yr = 4
            row_meta.append((label, color, t_yr, v_yr, te_yr, lag))
            continue
        # ── END BASELINE ROW ──────────────────────────────────────────────────

        # ── MODEL ROW ─────────────────────────────────────────────────────────
        try:
            (model, meta, bias, t_yr, v_yr, te_yr,
             tr_ld, va_ld, te_ld) = load_run(*run_spec)
        except Exception as exc:
            print(f'  Load failed: {exc}')
            for c in range(n_cols):
                _blank_ax(fig, cell_bboxes[(row_idx, c)], 'LOAD ERR')
            row_meta.append((label, color, 24, 8, 4, 0))
            continue

        row_meta.append((label, color, t_yr, v_yr, te_yr, meta.get('lag', 0)))

        run_lag      = meta.get('lag', 0)
        eff_year_idx = (target_year_idx - run_lag
                        if target_year_idx is not None else None)
        print('  Finding representative test sample ...')
        inp_np, tgt_log, pred_log = get_test_sample(
            te_ld, model, bias, valid_mask,
            target_year_idx=eff_year_idx,
        )

        if inp_np is None:
            print('  No valid test sample found; drawing blanks.')
            for c in range(n_cols):
                _blank_ax(fig, cell_bboxes[(row_idx, c)], 'NO DATA')
        else:
            ch = extract_channels(inp_np, meta)

            def arr_of(lst, single=None):
                items = list(lst or []) + ([single] if single is not None else [])
                items = [x for x in items if x is not None]
                return np.stack(items) if items else None

            def safe_range(arr, symmetric=False):
                if arr is None:
                    return (0.0, 8.0)
                lo, hi = float(arr.min()), float(arr.max())
                if symmetric:
                    ab = max(abs(lo), abs(hi)) or 1.0
                    return (-ab, ab)
                return (lo, hi)

            sp_arr  = arr_of(ch['sp_hist'],  ch['sp_curr'])
            rh_arr  = arr_of(ch['rec_hist'])
            tmp_arr = arr_of(ch['temp_hist'], ch['temp_curr'])
            rec_arr = arr_of([tgt_log, pred_log])

            sp_vmin,  sp_vmax  = safe_range(sp_arr)
            rh_vmin,  rh_vmax  = safe_range(rh_arr)
            tmp_vmin, tmp_vmax = safe_range(tmp_arr, symmetric=True)
            rec_vmin, rec_vmax = safe_range(rec_arr)

            # Col 0 — spawner history
            if ch['sp_hist']:
                draw_stack_ax(fig, cell_bboxes[(row_idx, 0)],
                              ch['sp_hist'], CMAPS['spawner'],
                              sp_vmin, sp_vmax, color, valid_mask)
            else:
                _blank_ax(fig, cell_bboxes[(row_idx, 0)], 'not used')

            # Col 1 — spawner current
            if ch['sp_curr'] is not None:
                draw_single_ax(fig, cell_bboxes[(row_idx, 1)],
                               ch['sp_curr'], CMAPS['spawner'],
                               sp_vmin, sp_vmax, color, valid_mask)
            elif pred_mode == 'normal':
                _blank_ax(fig, cell_bboxes[(row_idx, 1)], 'not used')
            else:
                _blank_ax(fig, cell_bboxes[(row_idx, 1)], 'excluded\n(forecast mode)')

            # Col 2 — temp history
            if ch['temp_hist']:
                draw_stack_ax(fig, cell_bboxes[(row_idx, 2)],
                              ch['temp_hist'], CMAPS['temp'],
                              tmp_vmin, tmp_vmax, color, valid_mask)
            else:
                _blank_ax(fig, cell_bboxes[(row_idx, 2)], 'not used')

            # Col 3 — temp current
            if ch['temp_curr'] is not None:
                draw_single_ax(fig, cell_bboxes[(row_idx, 3)],
                               ch['temp_curr'], CMAPS['temp'],
                               tmp_vmin, tmp_vmax, color, valid_mask)
            elif pred_mode == 'one_year_ahead':
                _blank_ax(fig, cell_bboxes[(row_idx, 3)], 'excluded\n(forecast mode)')
            else:
                _blank_ax(fig, cell_bboxes[(row_idx, 3)], 'not used')

            # Col 4 — recruit history
            if ch['rec_hist']:
                draw_stack_ax(fig, cell_bboxes[(row_idx, 4)],
                              ch['rec_hist'], CMAPS['recruit'],
                              rh_vmin, rh_vmax, color, valid_mask)
            else:
                _blank_ax(fig, cell_bboxes[(row_idx, 4)], 'not used')

            # Col 5 — observed target
            draw_single_ax(fig, cell_bboxes[(row_idx, 5)],
                           tgt_log, CMAPS['recruit'],
                           rec_vmin, rec_vmax, color, valid_mask)

            # Col 6 — predicted
            draw_single_ax(fig, cell_bboxes[(row_idx, 6)],
                           pred_log, CMAPS['recruit'],
                           rec_vmin, rec_vmax, color, valid_mask)

        print('  Collecting yearly aggregates ...')
        agg = collect_yearly_aggregates(
            [tr_ld, va_ld, te_ld], model, bias, valid_mask
        )
        line_data[row_idx] = agg

    # ── Section divider — drawn as top border of the gap row ─────────────────
    if LAG5_SECTION_START < n_rows:
        divider_y = gap_bboxes[0].y1   # top edge of the gap row

        fig.add_artist(Line2D(
            [inputs_x0, target_x1], [divider_y, divider_y],
            transform=fig.transFigure,
            color='#333333', lw=2.0, ls='-',
        ))


    # ── Section headers + vertical dividers ───────────────────────────────────
    _wbg = dict(facecolor='white', edgecolor='none', alpha=0.90, pad=2)

    for cx, lbl in [(inputs_cx, 'Inputs'),
                    (output_cx, 'Target'),
                    (target_cx, 'Output')]:
        fig.text(cx, sec_hdr_y, lbl,
                 ha='center', va='center', fontsize=FS_SECTION_HDR,
                 fontweight='bold', transform=fig.transFigure,
                 color='#111111', bbox=_wbg)

    for x0, x1 in [(inputs_x0, inputs_x1),
                   (output_x0, output_x1),
                   (target_x0, target_x1)]:
        fig.add_artist(Line2D([x0, x1], [rule_y, rule_y],
                              transform=fig.transFigure,
                              color='#aaaaaa', lw=1.0))

    for dx in [div_x, div2_x]:
        fig.add_artist(Line2D([dx, dx], [grid_bot, fig_hdr_top],
                              transform=fig.transFigure,
                              color='#666666', lw=1.5, ls='--'))

    # ── Line plot ──────────────────────────────────────────────────────────────
    print('\nBuilding line plot ...')
    obs_plotted = False
    for row_idx, (label, color, t_yr, v_yr, te_yr, lag) in enumerate(row_meta):
        if row_idx not in line_data:
            continue
        agg          = line_data[row_idx]
        years_sorted = sorted(agg.keys())
        years_plot   = [y + lag + DATA_START_YEAR for y in years_sorted]
        pred_med     = [np.median(agg[y]['pred']) for y in years_sorted]
        obs_med      = [np.median(agg[y]['obs'])  for y in years_sorted]

        is_baseline = (runs[row_idx][0] == 'BASELINE')

        if not is_baseline:
            pred_p25 = [np.percentile(agg[y]['pred'], 25) for y in years_sorted]
            pred_p75 = [np.percentile(agg[y]['pred'], 75) for y in years_sorted]
            ax_line.fill_between(years_plot, pred_p25, pred_p75,
                                 color=color, alpha=0.15)

        ax_line.plot(years_plot, pred_med, '-', color=color, lw=2.0,
                     label=f'{"Baseline" if is_baseline else "Pred"} — {label}')

        if not is_baseline and lag == 0 and not obs_plotted:
            ax_line.scatter(years_plot, obs_med, color='black', s=18, zorder=5,
                            alpha=0.85, label='Observed')
            obs_plotted = True

    # Phase shading
    if row_meta and line_data:
        _, _, t_yr, v_yr, te_yr, _ = row_meta[0]
        all_years = sorted({y for d in line_data.values() for y in d})
        if all_years:
            y0_cal        = all_years[0]  + DATA_START_YEAR
            y1_cal        = all_years[-1] + DATA_START_YEAR
            train_end_cal = t_yr          - 0.5 + DATA_START_YEAR
            val_end_cal   = t_yr + v_yr   - 0.5 + DATA_START_YEAR
            xform = ax_line.get_xaxis_transform()
            ax_line.axvspan(y0_cal,        train_end_cal, color='seagreen',   alpha=0.07)
            ax_line.axvspan(train_end_cal, val_end_cal,   color='darkorange', alpha=0.07)
            ax_line.axvspan(val_end_cal,   y1_cal,        color='crimson',    alpha=0.07)
            ax_line.axvline(train_end_cal, color='grey', lw=1.0, ls=':')
            ax_line.axvline(val_end_cal,   color='grey', lw=1.0, ls=':')
            ax_line.text((y0_cal + train_end_cal) / 2,      0.93, 'TRAIN',
                         transform=xform, ha='center', fontsize=FS_PHASE_LABEL,
                         color='seagreen', fontweight='bold')
            ax_line.text((train_end_cal + val_end_cal) / 2,  0.93, 'VALIDATION',
                         transform=xform, ha='center', fontsize=FS_PHASE_LABEL,
                         color='darkorange', fontweight='bold')
            ax_line.text((val_end_cal + y1_cal) / 2,         0.93, 'TEST',
                         transform=xform, ha='center', fontsize=FS_PHASE_LABEL,
                         color='crimson', fontweight='bold')

    # Legend — solid colour patches only, no dashed line entry
    # Strip newlines from labels so they appear on a single line in the legend
    legend_handles = [
        mpatches.Patch(color=color, label=label.replace('\n', ' ').strip()[:70])
        for label, color, *_ in row_meta
    ]
    if obs_plotted:
        legend_handles.append(
            plt.Line2D([0], [0], color='black', marker='o', ms=4,
                       lw=0, label='Observed')
        )
    ax_line.legend(handles=legend_handles, fontsize=FS_LEGEND,
                   loc='upper center', bbox_to_anchor=(0.5, -0.18),
                   framealpha=0.85, ncol=2, borderaxespad=0.)

    ax_line.set_xlabel('Year', fontsize=FS_AXIS_LABEL)
    ax_line.set_ylabel('Total recruit abundance (density)', fontsize=FS_AXIS_LABEL)
    ax_line.set_title(
        'Median Across Bootstrap Aggregate Yearly Recruits',
        fontsize=FS_AXIS_TITLE, fontweight='bold',
    )
    ax_line.grid(True, alpha=0.25)

    fig.suptitle(title, fontsize=FS_SUPTITLE, fontweight='bold', y=0.995)

    out = save_path or os.path.join(SAVE_DIR, 'top_model_figure.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'\nSaved -> {out}')
    return out


# ============================================================
#  ENTRY POINT
# ============================================================

if __name__ == '__main__':
    make_figure(display_year=2023)


[1/6]  Climatological baseline | Now-cast + 1-yr ahead
Loaded 100 bootstraps × 24 years | spawners=True  recruits=True  temp=True
Applying log1p scaling to spawners / recruits.
  in_channels=17  channel_mask_indices=[0, 1, 2, 3, 4, 5, 1, 2, 3, 4, 5, 0, 1, 2, 3, 4, 5]
Loaded 100 bootstraps × 8 years | spawners=True  recruits=True  temp=True
  Using 5 years of historical data from previous split
Applying log1p scaling to spawners / recruits.
  in_channels=17  channel_mask_indices=[0, 1, 2, 3, 4, 5, 1, 2, 3, 4, 5, 0, 1, 2, 3, 4, 5]
Loaded 100 bootstraps × 4 years | spawners=True  recruits=True  temp=True
  Using 5 years of historical data from previous split
Applying log1p scaling to spawners / recruits.
  in_channels=17  channel_mask_indices=[0, 1, 2, 3, 4, 5, 1, 2, 3, 4, 5, 0, 1, 2, 3, 4, 5]

[2/6]  All channels | Now-cast | MSE | Base-size
Loaded 100 bootstraps × 24 years | spawners=True  recruits=True  temp=True
Applying log1p scaling to spawners / recruits.
  in_channels=17  channel

In [41]:
"""
attribution_figure.py
=====================
Integrated Gradients attribution analysis for top CrabTransformer runs.

Creates three outputs per run:
  - Per-year spatial panels  (input channels + attributions side-by-side)
  - Grand-average spatial panel
  - Channel-attribution bar chart by year

Usage
-----
    python models/attribution_figure.py

Or import and call run_targeted_ig() from a Colab cell.
"""

import os, sys, json, math
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Paths ──────────────────────────────────────────────────────────────────────
REPO_DIR    = '/content/Teleconnections-ViT'
DRIVE_DIR   = '/content/drive/MyDrive/Teleconnection_ViT'
OUTPUTS_DIR = os.path.join(DRIVE_DIR, 'model_outputs')
SAVE_DIR    = os.path.join(DRIVE_DIR, 'analysis/attribution')

DATA_START_YEAR = 1988   # year_idx 0 = calendar year 1988

sys.path.insert(0, REPO_DIR)
from models.model     import CrabTransformer
from data.data_helper import get_dataloaders

os.makedirs(SAVE_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Font sizes ─────────────────────────────────────────────────────────────────
# Adjust these to resize all text in every figure produced by this script.
FS_TITLE      = 22   # suptitle / main figure title
FS_PANEL      = 16   # per-panel subplot titles  (channel labels in spatial maps)
FS_AXIS_LABEL = 18   # x / y axis labels in bar chart
FS_TICK       = 15   # tick labels in bar chart
FS_LEGEND     = 15   # legend entries

# ── Run display names ──────────────────────────────────────────────────────────
# Maps (model_size, level, channel_cfg, pred_mode, criterion) → display label.
# Edit entries here to control how each model is named in figure titles.
# Runs not listed here fall back to an auto-generated label.
RUN_DISPLAY_NAMES = {
    ('normal', 'real', 'all',        'normal',        'MSE'):     'All channels | Now-cast | MSE | Base-size',
    ('small',  'real', 'temp_only',  'normal',        'MSE'): 'Bottom Temp only | Now-cast | MSE | Reduced-size',
    ('small',  'real', 'rec_temp',   'one_year_ahead','MSE'):     'Recruits + Bottom Temp | 1-yr ahead | MSE | Reduced-size',
    ('normal', 'real', 'sp_temp',    'lag5',          'MSE'):     'Spawners + Bottom Temp | Lag-5 | MSE | Base-size',
}
# ── Target runs ────────────────────────────────────────────────────────────────
# (model_size, level, channel_cfg, pred_mode, criterion)
TARGET_MODELS = [
    ('normal', 'real', 'all',       'normal',        'MSE'),
    ('small',  'real', 'temp_only', 'normal',        'MSE'),
    ('small',  'real', 'rec_temp',  'one_year_ahead','MSE'),
    ('normal', 'real', 'sp_temp',   'lag5',          'MSE'),
]


# ── Channel name helper ────────────────────────────────────────────────────────

def get_active_channel_names(meta):
    """Return channel display names in the same order as the model's input channels."""
    lag   = meta.get('lag', 0)
    names = []
    incl  = meta['incl_curr']

    def _t(offset):
        """Format a time label relative to recruit year, e.g. lag=5 → t becomes t-5."""
        total = lag + offset
        return f't-{total}' if total > 0 else 't'

    if meta['use_spawners']:
        if incl:
            names.append(f'Spawner ({_t(0)})')
        names.extend([f'Spawner ({_t(k)})' for k in range(1, 6)])
    if meta['use_recruits']:
        names.extend([f'Recruit ({_t(k)})' for k in range(1, 6)])
    if meta['use_temp']:
        if incl:
            names.append(f'Bot. Temp ({_t(0)})')
        names.extend([f'Bot. Temp ({_t(k)})' for k in range(1, 6)])
    return names


# ── Step 1: Compute baseline ───────────────────────────────────────────────────

def compute_baseline(train_loader):
    print('  Computing baseline (mean training field)...')
    all_inputs = []
    for inputs, _, _, _, _, _ in train_loader:
        all_inputs.append(inputs)
    stacked  = torch.cat(all_inputs, dim=0)
    baseline = stacked.mean(dim=0, keepdim=True)
    print(f'    Baseline from {stacked.shape[0]} samples  '
          f'range [{baseline.min():.3f}, {baseline.max():.3f}]')
    return baseline


# ── Step 2: Organize samples by year ──────────────────────────────────────────

def collect_samples_by_year(loader):
    by_year = {}
    for inputs, targets, mask, year_idx, spat_mask, val_year in loader:
        for i in range(inputs.shape[0]):
            if val_year[i] == 0:
                continue
            yr = int(year_idx[i].item())
            by_year.setdefault(yr, []).append((
                inputs[i:i+1], targets[i:i+1], mask[i:i+1], year_idx[i:i+1],
            ))
    for yr in sorted(by_year.keys()):
        print(f'    Year idx {yr} ({yr + DATA_START_YEAR}): {len(by_year[yr])} bootstrap samples')
    return by_year


# ── Step 3: Integrated Gradients ──────────────────────────────────────────────

def integrated_gradients(model, actual_input, baseline, year_idx,
                         temporal_mask, n_steps=50, output_fn=None):
    if output_fn is None:
        output_fn = lambda out: out.mean()
    delta             = actual_input - baseline
    accumulated_grads = torch.zeros_like(actual_input)
    for step in range(n_steps):
        alpha        = step / n_steps
        interpolated = (baseline + alpha * delta).detach().clone().requires_grad_(True)
        output_fn(model(interpolated, year_idx, temporal_mask)).backward()
        accumulated_grads += interpolated.grad.detach()
        model.zero_grad()
    return ((accumulated_grads / n_steps) * delta).squeeze(0).cpu().numpy()


# ── Step 4: Per-year IG, averaged across bootstraps ───────────────────────────

def compute_yearly_attributions(model, by_year, baseline, n_steps=50, output_fn=None):
    baseline = baseline.to(DEVICE)
    yearly_attr, yearly_inputs = {}, {}
    for yr in sorted(by_year.keys()):
        samples = by_year[yr]
        print(f'\n  Year {yr} ({yr + DATA_START_YEAR}): running IG on {len(samples)} bootstraps...')
        attr_list, input_list = [], []
        for idx, (inp, _, msk, yr_idx) in enumerate(samples):
            attr = integrated_gradients(
                model, inp.to(DEVICE), baseline,
                yr_idx.to(DEVICE), msk.to(DEVICE),
                n_steps=n_steps, output_fn=output_fn,
            )
            attr_list.append(attr)
            input_list.append(inp.squeeze(0).numpy())
            if (idx + 1) % 25 == 0:
                print(f'    {idx + 1}/{len(samples)} done')
        yearly_attr[yr]   = np.stack(attr_list).mean(axis=0)
        yearly_inputs[yr] = np.stack(input_list).mean(axis=0)
        print(f'    Done. Attr range [{yearly_attr[yr].min():.6f}, {yearly_attr[yr].max():.6f}]')
    return yearly_attr, yearly_inputs


# ── Step 5: Visualization ─────────────────────────────────────────────────────

def plot_year_panel(yr, mean_input, mean_attr, meta, out_name, display_name):
    """
    Spatial panel: top half = mean input channels, bottom half = attributions.
    Grid width and height scale automatically to the number of channels.
    """
    mask_path  = os.path.join(REPO_DIR, 'data/real/output/spatial_mask.npy')
    valid_mask = (np.load(mask_path) > 0) if os.path.exists(mask_path) \
                 else np.ones((50, 50), dtype=bool)

    def _masked(image):
        return np.ma.array(image, mask=~valid_mask)

    import copy
    def _cmap(name):
        c = copy.copy(plt.get_cmap(name))
        c.set_bad('white')
        return c

    n_channels    = meta['in_channels']
    ch_names      = get_active_channel_names(meta)
    n_cols        = min(n_channels, 6)
    rows_input    = math.ceil(n_channels / n_cols)
    rows_attr     = math.ceil((n_channels + 1) / n_cols)
    total_rows    = rows_input + rows_attr

    cell_size = 4.0
    fig, axes = plt.subplots(
        total_rows, n_cols,
        figsize=(cell_size * n_cols, cell_size * total_rows),
        squeeze=False,
    )

    # Hide all panels; we'll re-enable only the ones that have content.
    for ax in axes.flat:
        ax.axis('off')

    abs_attr  = np.abs(mean_attr)
    vmax_attr = np.percentile(abs_attr, 99)

    # Top half — mean input channels
    for c in range(n_channels):
        row, col = divmod(c, n_cols)
        ax = axes[row, col]
        is_temp = 'Temp' in ch_names[c]
        cmap    = 'RdBu_r' if is_temp else 'viridis'
        vmax    = None if is_temp else 8.0
        ax.imshow(_masked(mean_input[c]), cmap=_cmap(cmap), vmin=None if is_temp else 0, vmax=vmax)
        ax.set_title(f'Input: {ch_names[c]}', fontsize=FS_PANEL, fontweight='bold')
        ax.axis('off')


    # Bottom half — attribution channels
    for c in range(n_channels):
        row, col = divmod(c, n_cols)
        ax = axes[rows_input + row, col]
        ax.imshow(_masked(mean_attr[c]), cmap=_cmap('RdBu_r'), vmin=-vmax_attr, vmax=vmax_attr)
        ax.set_title(f'Attr: {ch_names[c]}', fontsize=FS_PANEL, fontweight='bold')
        ax.axis('off')

    agg_row, agg_col = divmod(n_channels, n_cols)
    axes[rows_input + agg_row, agg_col].imshow(_masked(abs_attr.sum(axis=0)), cmap=_cmap('hot'))
    axes[rows_input + agg_row, agg_col].set_title(
        'Attr: Aggregate |IG|', fontsize=FS_PANEL, fontweight='bold')
    axes[rows_input + agg_row, agg_col].axis('off')

    lag    = meta.get('lag', 0)
    cal_yr = f'{yr + DATA_START_YEAR + lag}' if isinstance(yr, int) else 'Average'
    fig.suptitle(
        'Spatial Attribution\n' +
        f'{display_name} — {cal_yr}\n'
        'Top: Mean input across bootstraps  |  Bottom: Attribution  (red = +recruit, blue = −recruit)',
        fontsize=FS_TITLE, fontweight='bold',
    )
    plt.tight_layout(rect=[0, 0, 1, 0.94])

    path = os.path.join(SAVE_DIR, f'ig_year{yr}_{out_name}.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'  Saved: {path}')
    plt.close()


def plot_grand_average(yearly_attr, yearly_inputs, meta, out_name, display_name):
    years       = sorted(yearly_attr.keys())
    grand_attr  = np.stack([yearly_attr[yr]   for yr in years]).mean(axis=0)
    grand_input = np.stack([yearly_inputs[yr] for yr in years]).mean(axis=0)
    plot_year_panel('AVG', grand_input, grand_attr, meta, out_name, display_name)


def plot_channel_bar_yearly(yearly_attr, meta, out_name, display_name):
    """
    Bar chart: share of total |attribution| per channel, one bar group per year.
    Year indices are converted to calendar years in the legend.
    """
    years    = sorted(yearly_attr.keys())
    ch_names = get_active_channel_names(meta)
    n_ch     = meta['in_channels']

    # Build data: calendar-year string → % attribution array
    data = {}
    for yr in years:
        totals      = np.abs(yearly_attr[yr]).sum(axis=(1, 2))
        data[str(yr + DATA_START_YEAR)] = totals / totals.sum() * 100

    grand        = np.stack([yearly_attr[yr] for yr in years]).mean(axis=0)
    grand_totals = np.abs(grand).sum(axis=(1, 2))
    data['Average'] = grand_totals / grand_totals.sum() * 100

    x        = np.arange(n_ch)
    n_groups = len(data)
    width    = 0.8 / n_groups
    colors   = plt.cm.tab10(np.linspace(0, 1, n_groups))

    fig_w = max(16, 1.8 * n_ch)
    fig, ax = plt.subplots(figsize=(fig_w, 8))

    for i, (label, pcts) in enumerate(data.items()):
        offset = (i - n_groups / 2 + 0.5) * width
        ax.bar(x + offset, pcts, width, label=label,
               color=colors[i], alpha=0.8, edgecolor='black', linewidth=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(ch_names, rotation=45, ha='right', fontsize=FS_TICK)
    ax.tick_params(axis='y', labelsize=FS_TICK)
    ax.set_ylabel('Share of Total Attribution (%)', fontsize=FS_AXIS_LABEL)
    ax.set_title(
        f'Channel Attribution by Year\n{display_name}',
        fontsize=FS_TITLE, fontweight='bold',
    )
    ax.legend(loc='upper right', fontsize=FS_LEGEND)
    ax.grid(axis='y', alpha=0.3, ls='--')c
    plt.tight_layout()

    path = os.path.join(SAVE_DIR, f'ig_bar_yearly_{out_name}.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'  Saved: {path}')
    plt.close()


# ── Cache loader ──────────────────────────────────────────────────────────────

def load_saved_attributions(out_name):
    """
    Look for previously saved .npy attribution arrays in SAVE_DIR.

    Returns (yearly_attr, yearly_inputs) if all matching pairs are found,
    or (None, None) if no attr files exist for this run.
    """
    import glob
    prefix = 'ig_attr_yr'
    suffix = f'_{out_name}.npy'
    attr_paths = sorted(glob.glob(os.path.join(SAVE_DIR, f'{prefix}*{suffix}')))

    if not attr_paths:
        return None, None

    yearly_attr, yearly_inputs = {}, {}
    for attr_path in attr_paths:
        fname  = os.path.basename(attr_path)
        yr_str = fname[len(prefix):-len(suffix)]
        yr     = int(yr_str)

        input_path = os.path.join(SAVE_DIR, f'ig_input_yr{yr}{suffix}')
        if not os.path.exists(input_path):
            print(f'  Warning: attr file exists for year {yr} but input file is missing — will recompute.')
            return None, None

        yearly_attr[yr]   = np.load(attr_path)
        yearly_inputs[yr] = np.load(input_path)
        print(f'  Loaded year {yr} ({yr + DATA_START_YEAR})')

    return yearly_attr, yearly_inputs


# ── Main entry point ──────────────────────────────────────────────────────────

def run_targeted_ig(model_size, level, channel_cfg, pred_mode, criterion,
                    phase='TEST', n_steps=50, force_recompute=False):
    """
    Run (or reload) Integrated Gradients for one model configuration.

    Parameters
    ----------
    force_recompute : bool
        If False (default), saved .npy arrays in SAVE_DIR are reused and the
        expensive IG computation is skipped.  Set to True to redo everything
        from scratch (e.g. after changing n_steps or the model checkpoint).
    """
    run_spec     = (model_size, level, channel_cfg, pred_mode, criterion)
    out_name     = f'{model_size}_{level}_{channel_cfg}_{pred_mode}_{criterion}'
    display_name = RUN_DISPLAY_NAMES.get(
        run_spec,
        f'{channel_cfg} | {pred_mode} | {criterion} ({model_size})',
    )

    print(f'\n{"="*70}')
    print(f' IG attribution: {display_name}')
    print(f' Phase: {phase} | Steps: {n_steps} | force_recompute: {force_recompute}')
    print(f'{"="*70}')

    ckpt_dir     = os.path.join(OUTPUTS_DIR, model_size, level, channel_cfg, pred_mode, criterion)
    history_path = os.path.join(ckpt_dir, 'training_history.json')

    if not os.path.exists(history_path):
        print(f'  Missing training_history.json in {ckpt_dir} — skipping.')
        return

    # Meta is always loaded from JSON (fast — no model or data needed).
    with open(history_path) as f:
        meta = json.load(f)['channel_cfg_meta']
    print(f"  Channels: {meta['in_channels']}  Lag: {meta['lag']}  "
          f"Temp: {meta['use_temp']}  Recruits: {meta['use_recruits']}")

    # ── Try to use saved arrays ───────────────────────────────────────────────
    yearly_attr, yearly_inputs = None, None
    if not force_recompute:
        print('  Checking for saved attribution arrays...')
        yearly_attr, yearly_inputs = load_saved_attributions(out_name)
        if yearly_attr:
            print(f'  Found {len(yearly_attr)} saved years — skipping IG computation.')

    # ── Compute IG if no cached arrays ───────────────────────────────────────
    if yearly_attr is None:
        ckpt_path = os.path.join(ckpt_dir, 'best_model.pt')
        if not os.path.exists(ckpt_path):
            print(f'  Missing best_model.pt in {ckpt_dir} — skipping.')
            return

        model = CrabTransformer(
            grid_size=50, patch_size=5,
            in_channels=meta['in_channels'],
            embed_dim=meta['embed_dim'],
            num_heads=meta['num_heads'],
            num_layers=meta['num_layers'],
            d_ff=meta['d_ff'],
            dropout=0,
            channel_mask_indices=meta['channel_mask_indices'],
        ).to(DEVICE)
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        model.eval()

        t_yr, v_yr, te_yr = (24, 8, 4) if meta['lag'] == 0 else (21, 6, 4)
        train_loader, val_loader, test_loader = get_dataloaders(
            batch_size=8, memory_years=5,
            train_years=t_yr, val_years=v_yr, test_years=te_yr,
            data_type='real', level=level,
            include_current_spawner=meta['incl_curr'],
            lag=meta['lag'],
            use_temp=meta['use_temp'],
            use_spawners=meta['use_spawners'],
            use_recruits=meta['use_recruits'],
        )

        baseline      = compute_baseline(train_loader)
        target_loader = {'TEST': test_loader, 'VAL': val_loader, 'TRAIN': train_loader}[phase]

        print(f'\n  Collecting {phase} samples by year...')
        by_year = collect_samples_by_year(target_loader)

        yearly_attr, yearly_inputs = compute_yearly_attributions(
            model, by_year, baseline, n_steps=n_steps,
        )

        for yr in sorted(yearly_attr.keys()):
            np.save(os.path.join(SAVE_DIR, f'ig_attr_yr{yr}_{out_name}.npy'),   yearly_attr[yr])
            np.save(os.path.join(SAVE_DIR, f'ig_input_yr{yr}_{out_name}.npy'), yearly_inputs[yr])
        print(f'  Arrays saved to {SAVE_DIR}')

    # ── Regenerate figures (always runs) ─────────────────────────────────────
    print('\n  Generating visualizations...')
    for yr in sorted(yearly_attr.keys()):
        plot_year_panel(yr, yearly_inputs[yr], yearly_attr[yr], meta, out_name, display_name)

    plot_grand_average(yearly_attr, yearly_inputs, meta, out_name, display_name)
    plot_channel_bar_yearly(yearly_attr, meta, out_name, display_name)


# ── Entry point ───────────────────────────────────────────────────────────────

if __name__ == '__main__':
    # Set force_recompute=True to redo the IG computation from scratch.
    for run in TARGET_MODELS:
        run_targeted_ig(*run, phase='TEST', n_steps=50, force_recompute=False)



 IG attribution: All channels | Now-cast | MSE | Base-size
 Phase: TEST | Steps: 50 | force_recompute: False
  Channels: 17  Lag: 0  Temp: True  Recruits: True
  Checking for saved attribution arrays...
  Loaded year 33 (2021)
  Loaded year 34 (2022)
  Loaded year 35 (2023)
  Found 3 saved years — skipping IG computation.

  Generating visualizations...
  Saved: /content/drive/MyDrive/Teleconnection_ViT/analysis/attribution/ig_year33_normal_real_all_normal_MSE.png
  Saved: /content/drive/MyDrive/Teleconnection_ViT/analysis/attribution/ig_year34_normal_real_all_normal_MSE.png
  Saved: /content/drive/MyDrive/Teleconnection_ViT/analysis/attribution/ig_year35_normal_real_all_normal_MSE.png
  Saved: /content/drive/MyDrive/Teleconnection_ViT/analysis/attribution/ig_yearAVG_normal_real_all_normal_MSE.png
  Saved: /content/drive/MyDrive/Teleconnection_ViT/analysis/attribution/ig_bar_yearly_normal_real_all_normal_MSE.png

 IG attribution: Bottom Temp only | Now-cast | MSE | Reduced-size
 Phase